# Measure tissue thickness

Maps, per FOV, where (z, µm) real tissue signal starts and ends, from one
finished round (default `cells`), to see how much of each z-stack is wasted.

Procedure:
1. Resolve the round's frame table and `CHANNEL_NM`'s z-steps.
2. For every FOV, read every z-plane of `CHANNEL_NM` and cache an exact
   per-intensity histogram (Counter) of each frame
   (`analysis.fov.compute_channel_counters`). Later steps derive from it
   without re-reading pixels.
3. Find the highest-mean frame (tissue reference) and the
   `N_BACKGROUND_FRAMES` lowest-mean frames (background). `THRESHOLD` is the
   highest pixel value among the background frames. Check the plot and
   override it if needed.
4. Per FOV, derive the true-pixel-count (TPC) profile and report `z_first_um`
   and `z_last_um` (shallowest/deepest z with TPC > `TPC_THRESHOLD`) and
   `is_contiguous`.
5. Heatmaps of `z_first_um`/`z_last_um` on the FOV grid.
6. Verify `z_last_um`: a mosaic of each FOV's frame at its own last passing z
   (section 10). Every tile should look like a tissue edge.

Figures go to `SAMPLE_DIR/analysis/figures/`, results to `SAMPLE_DIR/analysis/`.

Runs on a cluster node too. Step 2's loop is sequential on purpose: a worker
pool sized from `os.cpu_count()` can exceed the job's memory and get
OOM-killed.

## 1 — Setup

In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from skimage.transform import resize as sk_resize
from scipy.ndimage import gaussian_filter, laplace, median_filter

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config      import ExperimentConfig
from MERci.common.metadata    import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.progress           import ProgressTracker
from MERci.progress_display   import ProgressReporter, format_duration
from MERci.common.io          import iter_image_frames, path_mtime
from MERci.analysis.fov       import (
    compute_channel_counters, save_channel_counters, load_channel_counters,
    counter_mean, counter_percentile, rebin_counter, tpc_profile_from_counters,
    atomic_save,
)
from MERci.analysis.round        import create_mosaic
from MERci.plots.experiment_plots         import display_mosaic
from MERci.scheduler             import resolve_round_flip_y
from MERci.acquisition.configs   import load_round_frame_table, read_hal_exposure_time
from MERci.acquisition.configs import load_microscope_orientation, apply_microscope_orientation
from MERci.plots.experiment_plots import get_merci_figures_dir

print(f"SAMPLE_DIR : {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX = ".zarr"   # must match what HAL wrote

# Microscope (e.g. "ST2"); selects the camera orientation for the section-10
# mosaic (resolve_microscope_parameters_filename).
MICROSCOPE = "ST2"

# Which round to use -- by imaging_type (default "cells"), or set ROUND_ID directly to override.
ROUND_IMAGING_TYPE = "cells"
ROUND_ID            = None

# Channel to measure tissue depth for.
CHANNEL_NM = 405.0

# A z-plane still counts as "has tissue" if its true-pixel count (TPC) exceeds this.
TPC_THRESHOLD = 1

# Heatmap color scale upper bound (micron).
MAX_Z_COLORMAP = 70.0

# Display-histogram resolution (section 5), re-binned from the exact Counters.
DISPLAY_HIST_BINS      = 200
LINEAR_HIST_PERCENTILE = 99.0   # upper bound of the linear-scale display histogram

# Lowest-mean frames treated as background for the threshold (section 5), and
# the percentile of each taken as its noise ceiling (100 = max).
N_BACKGROUND_FRAMES  = 10
BACKGROUND_PERCENTILE = 100.0

# Binarization threshold; None = highest pixel value among the background
# frames (section 5). Check that plot on a new experiment.
THRESHOLD = None

# Gaussian sigma (px) before the Laplacian-variance texture stat (section 14);
# suppresses pixel noise, keeps multi-pixel tissue structure.
TEXTURE_SMOOTH_SIGMA = 1.0

# Plot font sizes.
PLOT_TITLE_FONTSIZE    = 14
PLOT_LABEL_FONTSIZE    = 12
PLOT_TICK_FONTSIZE     = 11
PLOT_LEGEND_FONTSIZE   = 10
PLOT_SUPTITLE_FONTSIZE = 15

print(f"Sample name        : {SAMPLE_NAME}")
print(f"Positions tag       : {POSITIONS_TAG}")
print(f"Microscope         : {MICROSCOPE}")
print(f"Round imaging type : {ROUND_IMAGING_TYPE}  (ROUND_ID override: {ROUND_ID})")
print(f"Channel            : {CHANNEL_NM} nm")

In [ ]:
config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)

meta    = ExperimentMetadata.load(config.round_info_csv, config.positions_txt, config.data_dir,
                                   image_suffix=config.image_suffix)
tracker = ProgressTracker(config.analysis_dir)

figures_dir = get_merci_figures_dir(SAMPLE_DIR, "misc", NOTEBOOK_NAME)
figures_dir.mkdir(parents=True, exist_ok=True)

# Exact Counters are cached under analysis/cache/<notebook_name>/
# (FOVScheduler's 512-bin histograms can't be converted back).
NOTEBOOK_NAME = "measure_tissue_thickness"
cache_dir     = config.analysis_dir / "cache" / NOTEBOOK_NAME
channel_counters_dir = cache_dir / "channel_counters"
channel_counters_dir.mkdir(parents=True, exist_ok=True)

print(f"Rounds : {meta.n_rounds}")
print(f"FOVs   : {meta.n_fovs}")
print(f"Figures: {figures_dir}")
print(f"Cache  : {cache_dir}")

## 3 — Resolve the target round and its frame table

In [ ]:
target_round_id = ROUND_ID if ROUND_ID is not None else meta.round_for_imaging_type(ROUND_IMAGING_TYPE)
if not meta.round_fully_written(target_round_id):
    print(f"WARNING: round {target_round_id} is not yet fully written on disk -- "
          f"results below will be based on a partial FOV set.")

frame_table = load_round_frame_table(target_round_id, config, meta)

channel_frames = (
    frame_table[frame_table["color"].round(0) == round(CHANNEL_NM)]
    .sort_values("z")
)
if channel_frames.empty:
    raise ValueError(f"No frames found for channel {CHANNEL_NM} nm in round {target_round_id}'s frame table.")

z_frame_indices = list(zip(channel_frames.index.tolist(), channel_frames["z"].tolist()))

print(f"Target round : {target_round_id}")
print(f"Channel {CHANNEL_NM} nm has {len(z_frame_indices)} z-step(s) in this round's frame table.")

## 4 — Compute (or load cached) exact per-z Counter histograms for `CHANNEL_NM`

For every FOV, reads every z-plane of `CHANNEL_NM` and stores an exact
per-intensity Counter of each frame (`analysis.fov.compute_channel_counters`).
Sections 5, 6 and 11-13 derive from it. Cached in
`analysis/cache/measure_tissue_thickness/channel_counters/`.

In [ ]:
def channel_counters_path(fpath):
    return channel_counters_dir / f"{Path(fpath).stem}_counters.npz"


files = meta.files_for_round(target_round_id)
print(f"Round {target_round_id}: {len(files)} FOV file(s) expected.")

channel_counters = {}   # fov_id -> compute_channel_counters()-shaped dict
from_own_cache, to_compute = [], []
n_missing_on_disk = 0

for fpath in files:
    if channel_counters_path(fpath).exists():
        from_own_cache.append(fpath)
    elif fpath.exists():
        to_compute.append(fpath)
    else:
        n_missing_on_disk += 1

for fpath in from_own_cache:
    channel_counters[meta.fov_id_of_file(fpath)] = load_channel_counters(channel_counters_path(fpath))
print(f"{len(from_own_cache)} channel Counter(s) already cached -- loaded directly.")

n_computed = 0
if to_compute:
    print(f"Computing {len(to_compute)} missing channel Counter(s) sequentially "
          f"(all {len(z_frame_indices)} z-step(s) of channel {CHANNEL_NM:.0f} nm per FOV).")
    reporter = ProgressReporter(total=len(to_compute), label="Computing channel Counters")
    for fpath in reporter.wrap(to_compute):
        counters = compute_channel_counters(
            fpath, z_frame_indices,
            frame_width=config.frame_width, frame_height=config.frame_height,
        )
        save_channel_counters(channel_counters_path(fpath), counters)
        channel_counters[meta.fov_id_of_file(fpath)] = counters
        n_computed += 1

print(f"Channel Counters ready for {len(channel_counters)} / {len(files)} FOVs "
      f"({n_computed} newly computed this run, {n_missing_on_disk} not yet written on disk).")

## 5 — Reference frames: highest-mean (tissue) + lowest-mean (background) frames

Across every FOV and z: the highest-mean frame (what tissue looks like) and the
`N_BACKGROUND_FRAMES` lowest-mean frames (background). A fixed z pooled across
FOVs doesn't work, since it is blank in some FOVs.

`THRESHOLD` is the highest pixel value among those background frames.
Bounding only the background side keeps an empty frame's noise tail from
being called tissue. Check the overlaid histograms; if wrong, set
`THRESHOLD` in section 2 and re-run from here.

In [ ]:
# ---- Calculation --------------------------------------------------------
# Cached; replotting doesn't rescan while the Counter set is unchanged.
reference_frames_cache = cache_dir / f"reference_frames_round{target_round_id}.npz"
n_frames_now = sum(len(c["values_per_z"]) for c in channel_counters.values())

cached = None
if reference_frames_cache.exists():
    cached = np.load(reference_frames_cache)
    if int(cached["n_frames_considered"]) != n_frames_now:
        print(f"Cached reference-frame selection is stale "
              f"({int(cached['n_frames_considered'])} vs. {n_frames_now} frame(s) now available) -- recomputing.")
        cached = None

if cached is not None:
    n_frames_considered = int(cached["n_frames_considered"])
    best_mean, best_fov_id, best_pos, best_frame_idx, best_z_um = (
        float(cached["best_mean"]), int(cached["best_fov_id"]), int(cached["best_pos"]),
        int(cached["best_frame_idx"]), float(cached["best_z_um"]),
    )
    worst_fov_ids, worst_pos, worst_frame_idx, worst_z_um = (
        cached["worst_fov_ids"], cached["worst_pos"], cached["worst_frame_idx"], cached["worst_z_um"],
    )
    print(f"Loaded cached reference-frame selection ({reference_frames_cache.name}) -- "
          f"{n_frames_considered} FOV x z combination(s), matches current data.")
else:
    frame_records = []   # (mean, fov_id, pos_in_z, frame_idx, z_um)
    reporter = ProgressReporter(total=len(channel_counters), label="Scanning frame means")
    for fov_id, counters in reporter.wrap(channel_counters.items()):
        for pos, (values, counts) in enumerate(zip(counters["values_per_z"], counters["counts_per_z"])):
            mean = counter_mean(values, counts)
            frame_records.append((mean, fov_id, pos, int(counters["frame_indices"][pos]), float(counters["z_um"][pos])))

    n_frames_considered = len(frame_records)
    frame_records.sort(key=lambda r: r[0])
    best_mean, best_fov_id, best_pos, best_frame_idx, best_z_um = frame_records[-1]
    worst_records = frame_records[:N_BACKGROUND_FRAMES]

    worst_fov_ids   = np.array([r[1] for r in worst_records], dtype=np.int64)
    worst_pos       = np.array([r[2] for r in worst_records], dtype=np.int64)
    worst_frame_idx = np.array([r[3] for r in worst_records], dtype=np.int64)
    worst_z_um      = np.array([r[4] for r in worst_records], dtype=np.float64)

    np.savez_compressed(
        reference_frames_cache,
        n_frames_considered=n_frames_considered,
        best_mean=best_mean, best_fov_id=best_fov_id, best_pos=best_pos,
        best_frame_idx=best_frame_idx, best_z_um=best_z_um,
        worst_fov_ids=worst_fov_ids, worst_pos=worst_pos,
        worst_frame_idx=worst_frame_idx, worst_z_um=worst_z_um,
    )
    print(f"Scanned {n_frames_considered} FOV x z combination(s); cached selection to {reference_frames_cache.name}.")

best_values, best_counts = (channel_counters[best_fov_id]["values_per_z"][best_pos],
                            channel_counters[best_fov_id]["counts_per_z"][best_pos])
worst_value_counts = [
    (channel_counters[int(fov_id)]["values_per_z"][int(pos)], channel_counters[int(fov_id)]["counts_per_z"][int(pos)])
    for fov_id, pos in zip(worst_fov_ids, worst_pos)
]

# THRESHOLD estimate: highest BACKGROUND_PERCENTILE-th pixel value among the
# N_BACKGROUND_FRAMES lowest-mean frames.
estimated_threshold = float(max(
    counter_percentile(values, counts, BACKGROUND_PERCENTILE) for values, counts in worst_value_counts
))

print(f"Highest-mean frame: FOV {best_fov_id}, frame_idx={best_frame_idx}, z={best_z_um:.2f} um (mean {best_mean:.0f})")
print(f"Lowest-mean {len(worst_fov_ids)} frame(s) (background reference):")
for fov_id, frame_idx, z_um in zip(worst_fov_ids, worst_frame_idx, worst_z_um):
    print(f"  FOV {fov_id}, frame_idx={frame_idx}, z={z_um:.2f} um")
print(f"Estimated background noise ceiling (p{BACKGROUND_PERCENTILE:.1f} of {len(worst_value_counts)} "
      f"background frame(s)): {estimated_threshold:.0f}")

In [ ]:
# ---- Display --------------------------------------------------------------
# Counters have no spatial information -- re-read the tissue reference frame
# and the single emptiest background frame to display them.
best_fpath  = next(f for f in files if meta.fov_id_of_file(f) == best_fov_id)
worst_fpath = next(f for f in files if meta.fov_id_of_file(f) == int(worst_fov_ids[0]))
best_frame  = next(frame for _, frame in iter_image_frames(
    best_fpath, [best_frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
))
worst_frame = next(frame for _, frame in iter_image_frames(
    worst_fpath, [int(worst_frame_idx[0])], frame_width=config.frame_width, frame_height=config.frame_height,
))

fig_img, axes_img = plt.subplots(1, 2, figsize=(10, 5))
for ax, frame, fov_id, z_um, title in zip(
    axes_img, (worst_frame, best_frame), (int(worst_fov_ids[0]), best_fov_id), (float(worst_z_um[0]), best_z_um),
    ("Lowest-mean frame (background)", "Highest-mean frame (tissue)"),
):
    im = ax.imshow(frame, cmap="gray")
    ax.set_title(f"{title}\nFOV {fov_id}, z={z_um:.1f} um", fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    cbar = fig_img.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig_img.tight_layout()
fig_img.savefig(figures_dir / f"tissue_thickness_reference_frames_round{target_round_id}.png", dpi=150)
plt.show()

# Background histograms (thin) + tissue frame (bold), log and linear scale,
# re-binned from the Counters.
combined_min = float(min(min(v.min() for v, _ in worst_value_counts), best_values.min()))
combined_max = float(max(max(v.max() for v, _ in worst_value_counts), best_values.max()))
pct_value    = counter_percentile(best_values, best_counts, LINEAR_HIST_PERCENTILE)

log_edges       = np.logspace(np.log10(max(combined_min, 1)), np.log10(combined_max), DISPLAY_HIST_BINS + 1)
log_bin_centers = np.sqrt(log_edges[:-1] * log_edges[1:])   # geometric mean = correct center in log space

linear_edges       = np.linspace(combined_min, max(pct_value, combined_min + 1), DISPLAY_HIST_BINS + 1)
linear_bin_centers = 0.5 * (linear_edges[:-1] + linear_edges[1:])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for values, counts in worst_value_counts:
    axes[0].plot(log_bin_centers, rebin_counter(values, counts, log_edges), "-", color="steelblue", alpha=0.5, lw=1.0)
    axes[1].plot(linear_bin_centers, rebin_counter(values, counts, linear_edges), "-", color="steelblue", alpha=0.5, lw=1.0)
axes[0].plot(log_bin_centers, rebin_counter(best_values, best_counts, log_edges), "-", color="darkorange", lw=1.8)
axes[1].plot(linear_bin_centers, rebin_counter(best_values, best_counts, linear_edges), "-", color="darkorange", lw=1.8)
# One labeled proxy line per style -- a legend entry per background line would repeat N_BACKGROUND_FRAMES times.
for ax in axes:
    ax.plot([], [], "-", color="steelblue", alpha=0.5, lw=1.0,
            label=f"{len(worst_value_counts)} lowest-mean (background) frames")
    ax.plot([], [], "-", color="darkorange", lw=1.8, label=f"highest-mean frame (FOV {best_fov_id})")

axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_ylabel("Pixel count", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_title("Log-scale (full range)", fontsize=PLOT_TITLE_FONTSIZE)

axes[1].set_xlabel(f"Intensity  (channel {CHANNEL_NM:.0f} nm)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_ylabel("Pixel count", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_title(f"Linear-scale (min={combined_min:.0f} -> p{LINEAR_HIST_PERCENTILE:.0f}={pct_value:.0f})",
                   fontsize=PLOT_TITLE_FONTSIZE)

for ax in axes:
    ax.axvline(estimated_threshold, color="crimson", linestyle="--", lw=1.5,
               label=f"background noise ceiling = {estimated_threshold:.0f}")
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- {len(worst_value_counts)} lowest-mean (background) vs. "
             f"highest-mean (tissue) frame histograms", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()
fig.savefig(figures_dir / f"tissue_thickness_histogram_round{target_round_id}.png", dpi=150)
plt.show()

if THRESHOLD is None:
    THRESHOLD = estimated_threshold
print(f"Using THRESHOLD = {THRESHOLD:.0f}")

## 6 — Per-FOV: true-pixel-count (TPC) profile, derived from cached Counters

Per-z TPC against `THRESHOLD` from the cached Counters
(`analysis.fov.tpc_profile_from_counters`), no disk read. Cached, keyed on
`THRESHOLD`/`TPC_THRESHOLD`/FOV count. Reports `z_first_um`/`z_last_um`
(shallowest/deepest z with signal) and `is_contiguous` (`False` if signal
stops and restarts: debris, folds, noise).

In [ ]:
# ---- Calculation --------------------------------------------------------
# Cached, keyed on THRESHOLD/TPC_THRESHOLD/FOV count.
results_cache      = cache_dir / f"tpc_profile_round{target_round_id}.csv"
results_meta_cache = cache_dir / f"tpc_profile_round{target_round_id}.json"
cache_signature     = {"threshold": float(THRESHOLD), "tpc_threshold": float(TPC_THRESHOLD),
                        "n_fovs": len(channel_counters)}

cached_signature = json.loads(results_meta_cache.read_text()) if results_meta_cache.exists() else None

if cached_signature == cache_signature and results_cache.exists():
    results_df = pd.read_csv(results_cache)
    print(f"Loaded cached TPC profile ({results_cache.name}) -- "
          f"THRESHOLD/TPC_THRESHOLD/FOV count unchanged since it was written.")
else:
    results = []
    reporter = ProgressReporter(total=len(channel_counters), label="Deriving TPC profiles")
    for fov_id, counters in reporter.wrap(channel_counters.items()):
        profile = tpc_profile_from_counters(counters, THRESHOLD, TPC_THRESHOLD)
        results.append({
            "fov_id":        fov_id,
            "z_first_um":    profile["z_first_um"],
            "z_last_um":     profile["z_last_um"],
            "is_contiguous": profile["is_contiguous"],
            "x_um":          meta.fovs[fov_id].position[0],
            "y_um":          meta.fovs[fov_id].position[1],
        })
    results_df = pd.DataFrame(results)
    results_df.to_csv(results_cache, index=False)
    results_meta_cache.write_text(json.dumps(cache_signature))
    print(f"Derived TPC profile for {len(results_df)} FOV(s); cached to {results_cache.name}.")

In [ ]:
# ---- Display --------------------------------------------------------------
n_no_signal      = results_df["z_last_um"].isna().sum()
n_not_contiguous = (~results_df["is_contiguous"]).sum()
print(f"{len(results_df)} FOV(s) measured; {n_no_signal} had no z-plane above TPC_THRESHOLD at all; "
      f"{n_not_contiguous} had signal turn off and back on somewhere in between (is_contiguous=False).")
print("z_first_um:")
print(results_df["z_first_um"].describe())
print("z_last_um:")
print(results_df["z_last_um"].describe())

## 7 — Tissue-extent heatmaps across the FOV grid

Two panels on one colour scale: where signal **starts** (`z_first_um`) and
where it **ends** (`z_last_um`).

In [ ]:
# ---- Calculation --------------------------------------------------------
def positions_to_grid_indices(fov_ids, meta):
    """Stage (x, y) positions -> integer (x_idx, y_idx) grid indices (same approach as
    view_intensity_stats.ipynb's heatmap: round to the nearest integer micron, then
    rank each axis's unique values -- robust to float imprecision on a regular grid)."""
    xs = np.array([round(meta.fovs[f].position[0]) for f in fov_ids])
    ys = np.array([round(meta.fovs[f].position[1]) for f in fov_ids])
    unique_xs = np.sort(np.unique(xs))
    unique_ys = np.sort(np.unique(ys))
    x_rank = {v: i for i, v in enumerate(unique_xs)}
    y_rank = {v: i for i, v in enumerate(unique_ys)}
    return {f: (x_rank[xs[i]], y_rank[ys[i]]) for i, f in enumerate(fov_ids)}


def build_matrix(column, results_df, grid, n_x, n_y):
    matrix = np.full((n_y, n_x), np.nan)
    for _, row in results_df.iterrows():
        xi, yi = grid[row["fov_id"]]
        if pd.notna(row[column]):
            matrix[yi, xi] = row[column]
    return matrix


fov_ids = results_df["fov_id"].tolist()
grid    = positions_to_grid_indices(fov_ids, meta)
n_x     = max(xi for xi, _ in grid.values()) + 1
n_y     = max(yi for _, yi in grid.values()) + 1

z_first_matrix = build_matrix("z_first_um", results_df, grid, n_x, n_y)
z_last_matrix  = build_matrix("z_last_um",  results_df, grid, n_x, n_y)

print(f"FOV grid: {n_x} x {n_y} (columns x rows), {len(fov_ids)} FOV(s) placed.")

In [ ]:
# ---- Display --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(max(9, n_x * 0.7 + 3), max(4, n_y * 0.4 + 1.5)))
for ax, matrix, title in zip(
    axes,
    (z_first_matrix, z_last_matrix),
    ("z_first -- signal starts", "z_last -- signal ends"),
):
    im = ax.imshow(matrix, cmap="viridis", origin="upper", vmin=0, vmax=MAX_Z_COLORMAP)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.set_xlabel("X grid index  (increasing stage X â†’)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("Y grid index  (increasing stage Y â†“)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.04)
cbar.set_label("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.suptitle(f"Round {target_round_id} -- tissue extent map ({len(fov_ids)} FOVs)", fontsize=PLOT_SUPTITLE_FONTSIZE)

fig.savefig(figures_dir / f"tissue_thickness_heatmap_round{target_round_id}.png", dpi=150)
plt.show()

results_csv = config.analysis_dir / f"tissue_thickness_round{target_round_id}.csv"
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")

## 8 — Experimental vs. theoretical acquisition time per frame

- **Theoretical**: the round's HAL `<exposure_time>` only.
- **Experimental**: median time between consecutive FOV files finishing
  (`common.io.path_mtime`), divided by the round's frame count. Includes real
  overhead (stage moves, readout).

Section 9 uses the experimental rate. Cached, keyed on the FOV file count.

In [ ]:
# ---- Calculation --------------------------------------------------------
# Cached, keyed on the FOV file count (grows while acquiring).
timing_cache = cache_dir / f"timing_round{target_round_id}.json"

files_for_timing = [f for f in meta.files_for_round(target_round_id) if f.exists()]
if len(files_for_timing) < 2:
    raise ValueError(
        f"Need at least 2 written FOV files to measure real inter-FOV acquisition "
        f"timing -- only {len(files_for_timing)} found for round {target_round_id}."
    )

cached_timing = json.loads(timing_cache.read_text()) if timing_cache.exists() else None

if cached_timing is not None and cached_timing["n_files"] == len(files_for_timing):
    experimental_delta_s          = cached_timing["experimental_delta_s"]
    experimental_time_per_frame_s = cached_timing["experimental_time_per_frame_s"]
    theoretical_time_per_frame_s  = cached_timing["theoretical_time_per_frame_s"]
    delta_s = np.array(cached_timing["delta_s"])
    print(f"Loaded cached timing stats ({timing_cache.name}) -- "
          f"{len(files_for_timing)} FOV file(s), unchanged since it was written.")
else:
    reporter = ProgressReporter(total=len(files_for_timing), label="Reading FOV file mtimes")
    mtimes = [path_mtime(f) for f in reporter.wrap(files_for_timing)]
    mtimes = sorted(mtimes)
    delta_s = np.diff(mtimes)   # real wall-clock time between consecutive FOV-movies finishing

    # Median: robust to occasional long gaps.
    experimental_delta_s          = float(np.median(delta_s))
    experimental_time_per_frame_s = experimental_delta_s / len(frame_table)

    exposure_time_s = None
    for s in meta.series_for_round(target_round_id):
        if not s.hal_config:
            continue
        exp = read_hal_exposure_time(Path(config.settings_dir) / s.hal_config)
        if exp is not None:
            exposure_time_s = exp
            break
    if exposure_time_s is None:
        exposure_time_s = 0.25
        print("WARNING: could not read <exposure_time> from this round's HAL config -- "
              "falling back to 0.25 s/frame (same fallback acquisition.dave.estimate_dave_experiment uses).")
    theoretical_time_per_frame_s = exposure_time_s

    timing_cache.write_text(json.dumps({
        "n_files":                       len(files_for_timing),
        "delta_s":                       delta_s.tolist(),
        "experimental_delta_s":          experimental_delta_s,
        "experimental_time_per_frame_s": experimental_time_per_frame_s,
        "theoretical_time_per_frame_s":  theoretical_time_per_frame_s,
    }))
    print(f"Measured timing from {len(files_for_timing)} FOV file(s); cached to {timing_cache.name}.")

In [ ]:
# ---- Display --------------------------------------------------------------
print(f"Inter-FOV write-time delta (n={len(delta_s)} gaps, {len(files_for_timing)} FOV files): "
      f"median {experimental_delta_s:.2f}s, min {delta_s.min():.2f}s, max {delta_s.max():.2f}s, "
      f"std {delta_s.std():.2f}s")
print(f"Frames/FOV this round: {len(frame_table)}")
print(f"\nTheoretical  (HAL exposure_time only) : {theoretical_time_per_frame_s:.4f} s/frame")
print(f"Experimental (real file-write deltas)  : {experimental_time_per_frame_s:.4f} s/frame")
print(f"Experimental / theoretical ratio       : {experimental_time_per_frame_s / theoretical_time_per_frame_s:.2f}x "
      f"(> 1 means real per-frame time includes overhead the theoretical estimate misses)")
print("\n--> Using the EXPERIMENTAL rate for the time-savings estimate in section 9.")

## 9 — What-if: trim z-range acquisition

How much disk space and time would the round save if each FOV were imaged only
to `min(z_last_um + Z_MARGIN_UM, Z_MAX_TRIMMED_UM)`? No new image reads.

- **`Z_MAX_TRIMMED_UM`**: absolute cap on trimmed depth. Default: the deepest
  measured `z_last_um` + `Z_MARGIN_UM` (full depth if no FOV has signal), so it
  never cuts measured signal. The calculation warns if a value you set does.
- Every z-swept colour group (including blank/return frames) is assumed to
  scale with `CHANNEL_NM`'s cutoff. Fixed frames (e.g. one bead shot) are
  unaffected. FOVs with no signal need 0 z-swept frames.
- Savings are reported at both rates (theoretical and experimental), per round
  and per FOV file. Experimental is the one to plan with.
- `N_ROUNDS_LIKE_THIS` (default 1) extrapolates to the experiment; set it to
  the number of rounds with this z-sweep.

In [ ]:
# Extra margin (um) kept beyond each FOV's measured z_last_um.
Z_MARGIN_UM = 3.0

# Absolute cap (um) on trimmed depth. Default: deepest measured z_last_um +
# Z_MARGIN_UM, so it never cuts measured signal. Set lower only for a real
# limit; the next cell warns if it cuts any FOV's signal.
_z_last_um_max = results_df["z_last_um"].max()
Z_MAX_TRIMMED_UM = (
    float(_z_last_um_max + Z_MARGIN_UM) if pd.notna(_z_last_um_max)
    # No signal in any FOV: full imaged depth (no trimming).
    else float(frame_table["z"].max())
)

# Rounds sharing this z-sweep configuration (1 = this round only). Set by hand.
N_ROUNDS_LIKE_THIS = 1

print(f"Z_MARGIN_UM={Z_MARGIN_UM}, Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM:.1f} "
      f"(auto-derived from this round's own data -- override above for a different cap), "
      f"N_ROUNDS_LIKE_THIS={N_ROUNDS_LIKE_THIS}")

In [ ]:
# ---- Calculation --------------------------------------------------------
# Cached, keyed on the trim parameters.
trim_cache      = cache_dir / f"zrange_trim_round{target_round_id}.csv"
trim_meta_cache = cache_dir / f"zrange_trim_round{target_round_id}.json"
trim_signature  = {
    "z_margin_um": Z_MARGIN_UM, "z_max_trimmed_um": Z_MAX_TRIMMED_UM,
    "threshold": float(THRESHOLD), "tpc_threshold": float(TPC_THRESHOLD),
    "n_fovs": len(results_df),
}

cached_trim_signature = json.loads(trim_meta_cache.read_text()) if trim_meta_cache.exists() else None

if cached_trim_signature == trim_signature and trim_cache.exists():
    trim_df = pd.read_csv(trim_cache)
    print(f"Loaded cached trim table ({trim_cache.name}) -- trim parameters/THRESHOLD/FOV count unchanged.")
else:
    # Group frames by colour (blank frames get their own key). A group is
    # z-swept if its frames span more than one z.
    color_key = frame_table["color"].round(0)
    color_key = color_key.where(color_key.notna(), -1)

    zswept_groups = {}
    n_fixed_frames = 0
    for key, grp in frame_table.groupby(color_key):
        z_vals = grp["z"].to_numpy()
        if pd.Series(z_vals).nunique() > 1:
            zswept_groups[key] = z_vals
        else:
            n_fixed_frames += len(grp)

    n_zswept_frames = sum(len(v) for v in zswept_groups.values())
    print(f"Frame table: {len(frame_table)} frame(s)/FOV total -- {len(zswept_groups)} z-swept color "
          f"group(s) ({n_zswept_frames} frame(s)), {n_fixed_frames} fixed frame(s) unaffected by trimming.")

    def frames_kept_for_fov(z_needed):
        """Frame count kept under the trimmed scheme: every fixed frame, plus every
        z-swept-group frame at or below z_needed (0 z-swept frames if z_needed is None,
        i.e. no signal was detected in this FOV at all)."""
        if z_needed is None:
            return n_fixed_frames
        return n_fixed_frames + sum(int((z_vals <= z_needed).sum()) for z_vals in zswept_groups.values())

    trim_rows = []
    reporter = ProgressReporter(total=len(results_df), label="Computing per-FOV trim")
    for _, row in reporter.wrap(list(results_df.iterrows())):
        z_last   = row["z_last_um"]
        z_needed = min(z_last + Z_MARGIN_UM, Z_MAX_TRIMMED_UM) if pd.notna(z_last) else None
        n_trimmed = frames_kept_for_fov(z_needed)
        trim_rows.append({
            "fov_id":            row["fov_id"],
            "z_last_um":         z_last,
            "z_needed_um":       z_needed,
            "n_frames_current":  len(frame_table),
            "n_frames_trimmed":  n_trimmed,
            "n_frames_removed":  len(frame_table) - n_trimmed,
        })
    trim_df = pd.DataFrame(trim_rows)
    trim_df.to_csv(trim_cache, index=False)
    trim_meta_cache.write_text(json.dumps(trim_signature))
    print(f"Computed trim table for {len(trim_df)} FOV(s); cached to {trim_cache.name}.")

# Warn whenever Z_MAX_TRIMMED_UM cuts a FOV's measured signal.
_capped = (trim_df["z_last_um"] + Z_MARGIN_UM) > Z_MAX_TRIMMED_UM
n_capped = int(_capped.sum())
if n_capped > 0:
    max_truncated_um = float((trim_df["z_last_um"] + Z_MARGIN_UM - Z_MAX_TRIMMED_UM).clip(lower=0).max())
    print(f"\nWARNING: Z_MAX_TRIMMED_UM={Z_MAX_TRIMMED_UM:.1f} um is BELOW z_last_um+Z_MARGIN_UM for "
          f"{n_capped}/{len(trim_df)} FOV(s) -- the trimmed scheme would cut off up to "
          f"{max_truncated_um:.1f} um of real measured tissue signal for those FOV(s). "
          f"Raise Z_MAX_TRIMMED_UM in the parameters cell above if this isn't intentional.")

In [ ]:
# ---- Display --------------------------------------------------------------
def format_bytes(n):
    n = float(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if abs(n) < 1024 or unit == "TiB":
            return f"{n:.2f} {unit}"
        n /= 1024


# Frame size from one real frame (config.frame_width/height is often unset
# for .zarr/.tiff).
_sample_fpath = next(f for f in files if f.exists())
_sample_frame = next(frame for _, frame in iter_image_frames(
    _sample_fpath, [0], frame_width=config.frame_width, frame_height=config.frame_height,
))
frame_height_px, frame_width_px = _sample_frame.shape
frame_bytes = frame_width_px * frame_height_px * 2   # uint16 -- 2 bytes/pixel

n_fovs                          = len(trim_df)
bytes_saved_per_fov              = trim_df["n_frames_removed"] * frame_bytes
total_bytes_current_this_round   = n_fovs * len(frame_table) * frame_bytes
total_bytes_saved_this_round     = int(bytes_saved_per_fov.sum())

print(f"\n--- Round {target_round_id}, {n_fovs} FOV(s) ---")
print(f"Frames/FOV: {len(frame_table)} -> mean {trim_df['n_frames_trimmed'].mean():.1f} "
      f"({trim_df['n_frames_removed'].mean():.1f} removed/FOV on average)")
print(f"Space: {format_bytes(total_bytes_current_this_round)} -> "
      f"{format_bytes(total_bytes_current_this_round - total_bytes_saved_this_round)}  "
      f"(saved {format_bytes(total_bytes_saved_this_round)}, "
      f"{100 * total_bytes_saved_this_round / total_bytes_current_this_round:.1f}%)")

# Savings at the theoretical and experimental rates, per round and per FOV file.
time_rates = {
    "theoretical": theoretical_time_per_frame_s,
    "experimental": experimental_time_per_frame_s,
}
time_saved_per_fov_s = {}   # label -> pd.Series, one value per FOV
total_time_current_s = {}
total_time_saved_s   = {}

for label, rate in time_rates.items():
    time_saved_per_fov_s[label] = trim_df["n_frames_removed"] * rate
    total_time_current_s[label] = n_fovs * len(frame_table) * rate
    total_time_saved_s[label]   = float(time_saved_per_fov_s[label].sum())

    print(f"\nTime ({label}, {rate:.4f} s/frame):")
    print(f"  Round total:  {format_duration(total_time_current_s[label])} -> "
          f"{format_duration(total_time_current_s[label] - total_time_saved_s[label])}  "
          f"(saved {format_duration(total_time_saved_s[label])}, "
          f"{100 * total_time_saved_s[label] / total_time_current_s[label]:.1f}%)")
    print(f"  Single FOV file (average): {trim_df['n_frames_removed'].mean():.1f} frame(s) removed x "
          f"{rate:.4f} s/frame = {format_duration(time_saved_per_fov_s[label].mean())} saved")

print(f"\n--> Experimental is the recommended estimate for planning: it already reflects real "
      f"per-frame overhead (stage/z-move, fluidics, camera readout, ...) the theoretical "
      f"exposure-time-only rate misses.")

if N_ROUNDS_LIKE_THIS != 1:
    print(f"\n--- Extrapolated to {N_ROUNDS_LIKE_THIS} round(s) assumed to share this z-sweep "
          f"(N_ROUNDS_LIKE_THIS) ---")
    print(f"Space saved: {format_bytes(total_bytes_saved_this_round * N_ROUNDS_LIKE_THIS)}")
    for label in time_rates:
        print(f"Time saved ({label}): {format_duration(total_time_saved_s[label] * N_ROUNDS_LIKE_THIS)}")

trim_csv = config.analysis_dir / f"tissue_thickness_zrange_trim_round{target_round_id}.csv"
trim_df.to_csv(trim_csv, index=False)
print(f"\nSaved: {trim_csv}")

## 10 — Verify: per-FOV thumbnail mosaic at the last passing z

Each FOV's actual frame at its own `z_last_um`, tiled by stage position
(`analysis.round.create_mosaic`):

- Frames are re-oriented to the stage with the microscope's MERlin YAML
  (`MICROSCOPE`, `configs.apply_microscope_orientation`).
- One intensity scale pooled across all FOVs, not per tile.
- Each tile is labelled with its `z_last_um`.

Every tile should look like a tissue edge. A blank or clearly mid-tissue tile
points to a problem with the TPC threshold.

In [ ]:
# ---- Calculation --------------------------------------------------------
# Microscope camera->stage orientation, applied before downsampling.
MICROSCOPE_ORIENTATION_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "microscope"
MICROSCOPE_ORIENTATION = load_microscope_orientation(MICROSCOPE, MICROSCOPE_ORIENTATION_DIR)
print(f"Microscope orientation ({MICROSCOPE}): {MICROSCOPE_ORIENTATION}")

# Cached raw (not contrast-stretched) thumbnails, one per FOV, so all tiles
# share one intensity scale.
last_z_raw_thumbnails_dir = cache_dir / "last_z_raw_thumbnails" / f"round{target_round_id}"
last_z_raw_thumbnails_dir.mkdir(parents=True, exist_ok=True)


def last_z_raw_thumbnail_path(fov_id):
    return last_z_raw_thumbnails_dir / f"fov{fov_id:04d}.npy"


# A truncated cache file (killed job) is treated as missing.
def _npy_cache_valid(path):
    if not path.exists():
        return False
    try:
        np.load(path)
        return True
    except Exception:
        return False


fov_ids_with_signal = results_df.loc[results_df["z_last_um"].notna(), "fov_id"].tolist()
to_render = [f for f in fov_ids_with_signal if not _npy_cache_valid(last_z_raw_thumbnail_path(f))]
print(f"{len(fov_ids_with_signal)} FOV(s) with detected signal; "
      f"{len(fov_ids_with_signal) - len(to_render)} thumbnail(s) already cached, "
      f"{len(to_render)} to render.")

if to_render:
    tw, th = config.thumbnail_size
    reporter = ProgressReporter(total=len(to_render), label="Rendering last-z raw thumbnails")
    for fov_id in reporter.wrap(to_render):
        z_last    = float(results_df.loc[results_df["fov_id"] == fov_id, "z_last_um"].iloc[0])
        counters  = channel_counters[fov_id]
        pos       = int(np.argmin(np.abs(counters["z_um"] - z_last)))
        frame_idx = int(counters["frame_indices"][pos])

        fpath = next(f for f in files if meta.fov_id_of_file(f) == fov_id)
        frame = next(frame for _, frame in iter_image_frames(
            fpath, [frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
        ))
        frame = apply_microscope_orientation(frame, **MICROSCOPE_ORIENTATION)
        thumb = sk_resize(frame.astype(np.float64), (th, tw), anti_aliasing=True, preserve_range=True)
        atomic_save(last_z_raw_thumbnail_path(fov_id), lambda tmp: np.save(tmp, thumb.astype(np.float32)))

In [ ]:
# ---- Display --------------------------------------------------------------
raw_thumbnails = {fov_id: np.load(last_z_raw_thumbnail_path(fov_id)) for fov_id in fov_ids_with_signal}

# One shared intensity scale across all FOVs (thumbnail_percentile_clip).
pooled_pixels  = np.concatenate([t.ravel() for t in raw_thumbnails.values()])
lo_pct, hi_pct = config.thumbnail_percentile_clip
vmin, vmax     = np.percentile(pooled_pixels, [lo_pct, hi_pct])
print(f"Shared display scale (p{lo_pct:.0f}-p{hi_pct:.0f} over all {len(raw_thumbnails)} FOV thumbnails): "
      f"[{vmin:.0f}, {vmax:.0f}]")


def _to_uint8(thumb, vmin, vmax):
    scaled = (thumb.astype(np.float64) - vmin) / max(vmax - vmin, 1e-9) * 255
    return np.clip(scaled, 0, 255).astype(np.uint8)


last_z_thumbnails_uint8 = {fov_id: _to_uint8(t, vmin, vmax) for fov_id, t in raw_thumbnails.items()}
last_z_positions        = {fov_id: meta.fovs[fov_id].position for fov_id in fov_ids_with_signal}
last_z_labels           = {
    fov_id: f"{float(results_df.loc[results_df['fov_id'] == fov_id, 'z_last_um'].iloc[0]):.0f}"
    for fov_id in fov_ids_with_signal
}

last_z_mosaic_flip_y = resolve_round_flip_y(target_round_id, config, meta)
last_z_mosaic_path   = figures_dir / f"tissue_thickness_last_z_mosaic_round{target_round_id}.png"
create_mosaic(last_z_thumbnails_uint8, last_z_positions, last_z_mosaic_path,
              thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding,
              flip_y=last_z_mosaic_flip_y, labels=last_z_labels)
display_mosaic(last_z_mosaic_path, target_round_id)

n_missing_signal = len(results_df) - len(fov_ids_with_signal)
if n_missing_signal:
    print(f"({n_missing_signal} FOV(s) had no detected signal at all -- excluded from this "
          f"mosaic; see section 6's summary.)")

## 11 — Data exploration: per-FOV z-intensity profiles overlaid

Independent of `THRESHOLD`: each FOV's per-z median/max/min intensity from the
cached Counters, one thin line per FOV, in three subplots.

In [ ]:
# ---- Calculation --------------------------------------------------------
# Per-FOV, per-z median/max/min intensity from the cached Counters.
z_intensity_profile_cache = cache_dir / f"z_intensity_profile_round{target_round_id}.npz"
fov_id_list = list(channel_counters.keys())
n_z         = len(z_frame_indices)
z_axis_um   = np.array([z for _, z in z_frame_indices])

# Check required keys too: an older cache had different keys under the same name.
_EXPECTED_KEYS = {"median_intensity_matrix", "max_intensity_matrix", "min_intensity_matrix", "n_fovs", "n_z"}

cached_profile = None
if z_intensity_profile_cache.exists():
    cached_profile = np.load(z_intensity_profile_cache)
    if not _EXPECTED_KEYS.issubset(cached_profile.files):
        print("Cached per-z intensity profile has an outdated schema -- recomputing.")
        cached_profile = None
    elif int(cached_profile["n_fovs"]) != len(fov_id_list) or int(cached_profile["n_z"]) != n_z:
        print("Cached per-z intensity profile is stale (FOV/z count changed) -- recomputing.")
        cached_profile = None

if cached_profile is not None:
    median_intensity_matrix = cached_profile["median_intensity_matrix"]
    max_intensity_matrix    = cached_profile["max_intensity_matrix"]
    min_intensity_matrix    = cached_profile["min_intensity_matrix"]
    print(f"Loaded cached per-FOV z-intensity profiles ({z_intensity_profile_cache.name}).")
else:
    median_intensity_matrix = np.full((len(fov_id_list), n_z), np.nan)
    max_intensity_matrix    = np.full((len(fov_id_list), n_z), np.nan)
    min_intensity_matrix    = np.full((len(fov_id_list), n_z), np.nan)
    reporter = ProgressReporter(total=len(fov_id_list), label="Computing per-FOV z-intensity profiles")
    for i, fov_id in reporter.wrap(list(enumerate(fov_id_list))):
        counters = channel_counters[fov_id]
        for pos in range(len(counters["z_um"])):
            values, counts = counters["values_per_z"][pos], counters["counts_per_z"][pos]
            median_intensity_matrix[i, pos] = counter_percentile(values, counts, 50.0)
            max_intensity_matrix[i, pos]    = values.max()
            min_intensity_matrix[i, pos]    = values.min()

    np.savez_compressed(
        z_intensity_profile_cache,
        median_intensity_matrix=median_intensity_matrix,
        max_intensity_matrix=max_intensity_matrix,
        min_intensity_matrix=min_intensity_matrix,
        n_fovs=len(fov_id_list), n_z=n_z,
    )
    print(f"Computed z-intensity profiles for {len(fov_id_list)} FOV(s) x {n_z} z-step(s); "
          f"cached to {z_intensity_profile_cache.name}.")

In [ ]:
# ---- Display --------------------------------------------------------------
# One thin line per FOV; all three subplots share one y-axis.
shared_ymax = float(np.nanmax([median_intensity_matrix, max_intensity_matrix, min_intensity_matrix]))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
for ax, matrix, title in zip(
    axes,
    (median_intensity_matrix, max_intensity_matrix, min_intensity_matrix),
    ("Median intensity (per FOV)", "Max intensity (per FOV)", "Min intensity (per FOV)"),
):
    for row in matrix:
        ax.plot(z_axis_um, row, "-", color="steelblue", alpha=0.5, lw=1.0)
    ax.set_ylim(0, shared_ymax)
    ax.set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel(f"Intensity (channel {CHANNEL_NM:.0f} nm)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.suptitle(f"Round {target_round_id} -- per-FOV z-intensity profiles overlaid ({len(fov_id_list)} FOVs)",
             fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

z_intensity_profile_fig_path = figures_dir / f"tissue_thickness_z_intensity_profile_round{target_round_id}.png"
fig.savefig(z_intensity_profile_fig_path, dpi=150)
plt.show()
print(f"Saved: {z_intensity_profile_fig_path}")

## 12 — Same per-FOV z-intensity profiles as a heatmap, sorted by first-z value

Section 11's profiles as heatmaps (x = z, y = FOV). Each panel sorts FOVs by
its own first-z value and has its own colour scale.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True)
for ax, matrix, title in zip(
    axes,
    (median_intensity_matrix, max_intensity_matrix, min_intensity_matrix),
    ("Median intensity (per FOV)", "Max intensity (per FOV)", "Min intensity (per FOV)"),
):
    # Sort rows by first-z value, per panel.
    order         = np.argsort(matrix[:, 0])
    sorted_matrix = matrix[order]

    im = ax.imshow(
        sorted_matrix, aspect="auto", cmap="viridis",
        extent=[z_axis_um.min(), z_axis_um.max(), len(order), 0],
    )
    ax.set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("FOV rank (sorted by first-z value)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_title(title, fontsize=PLOT_TITLE_FONTSIZE)
    ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.suptitle(f"Round {target_round_id} -- per-FOV z-intensity profiles as heatmaps, "
             f"sorted by first-z value ({len(fov_id_list)} FOVs)", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

z_intensity_heatmap_fig_path = figures_dir / f"tissue_thickness_z_intensity_heatmap_round{target_round_id}.png"
fig.savefig(z_intensity_heatmap_fig_path, dpi=150)
plt.show()
print(f"Saved: {z_intensity_heatmap_fig_path}")

## 13 — Per-FOV TPC (true-pixel-count) profiles: lines + heatmap

Per-z TPC above `THRESHOLD` for every FOV, from the cached Counters: overlaid
lines (left) and a heatmap sorted by first-z value (right).

In [ ]:
# ---- Calculation --------------------------------------------------------
# Per-FOV TPC profiles from the cached Counters; cached, keyed on
# THRESHOLD/TPC_THRESHOLD/FOV/z count.
tpc_profile_cache      = cache_dir / f"tpc_profile_matrix_round{target_round_id}.npz"
tpc_profile_meta_cache = cache_dir / f"tpc_profile_matrix_round{target_round_id}.json"
tpc_profile_signature  = {"threshold": float(THRESHOLD), "tpc_threshold": float(TPC_THRESHOLD),
                          "n_fovs": len(fov_id_list), "n_z": n_z}

cached_tpc_signature = json.loads(tpc_profile_meta_cache.read_text()) if tpc_profile_meta_cache.exists() else None

if cached_tpc_signature == tpc_profile_signature and tpc_profile_cache.exists():
    tpc_matrix = np.load(tpc_profile_cache)["tpc_matrix"]
    print(f"Loaded cached TPC profile matrix ({tpc_profile_cache.name}).")
else:
    tpc_matrix = np.full((len(fov_id_list), n_z), np.nan)
    reporter = ProgressReporter(total=len(fov_id_list), label="Computing per-FOV TPC profiles")
    for i, fov_id in reporter.wrap(list(enumerate(fov_id_list))):
        profile = tpc_profile_from_counters(channel_counters[fov_id], THRESHOLD, TPC_THRESHOLD)
        tpc_matrix[i, :] = profile["tpc"]

    np.savez_compressed(tpc_profile_cache, tpc_matrix=tpc_matrix)
    tpc_profile_meta_cache.write_text(json.dumps(tpc_profile_signature))
    print(f"Computed TPC profile matrix for {len(fov_id_list)} FOV(s) x {n_z} z-step(s); "
          f"cached to {tpc_profile_cache.name}.")

In [ ]:
# ---- Display --------------------------------------------------------------
order             = np.argsort(tpc_matrix[:, 0])
sorted_tpc_matrix = tpc_matrix[order]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for row in tpc_matrix:
    axes[0].plot(z_axis_um, row, "-", color="steelblue", alpha=0.5, lw=1.0)
axes[0].axhline(TPC_THRESHOLD, color="crimson", linestyle="--", lw=1.5,
                label=f"TPC_THRESHOLD = {TPC_THRESHOLD}")
axes[0].set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_ylabel("TPC (true-pixel count)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_title("Per-FOV TPC profiles (overlaid)", fontsize=PLOT_TITLE_FONTSIZE)
axes[0].tick_params(labelsize=PLOT_TICK_FONTSIZE)
axes[0].legend(fontsize=PLOT_LEGEND_FONTSIZE)

im = axes[1].imshow(
    sorted_tpc_matrix, aspect="auto", cmap="viridis",
    extent=[z_axis_um.min(), z_axis_um.max(), len(order), 0],
)
axes[1].set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_ylabel("FOV rank (sorted by first-z value)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_title("Per-FOV TPC profiles (heatmap, sorted)", fontsize=PLOT_TITLE_FONTSIZE)
axes[1].tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("TPC", fontsize=PLOT_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- per-FOV TPC (true-pixel count) profiles ({len(fov_id_list)} FOVs)",
             fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

tpc_profile_fig_path = figures_dir / f"tissue_thickness_tpc_profile_round{target_round_id}.png"
fig.savefig(tpc_profile_fig_path, dpi=150)
plt.show()
print(f"Saved: {tpc_profile_fig_path}")

## 14 — Texture-based background/tissue discriminator: smoothed-Laplacian variance per FOV per z

An alternative to `THRESHOLD`/TPC: tissue has spatial structure, background
doesn't. Needs the 2D frames, so it has its own per-FOV cache
(`analysis/cache/measure_tissue_thickness/texture_stats/`).

Raw Laplacian variance is inflated by pixel noise; a Gaussian pre-blur
(`TEXTURE_SMOOTH_SIGMA`) removes it and keeps multi-pixel structure.

**Slow**: re-reads every FOV's full z-stack. `USE_SLURM_ARRAY = True` submits
one array task per missing FOV (`build_texture_stats_array_script` +
`cli_compute_texture_stats.py`); re-run the cell afterwards to load the
results. Safe to re-run.

Exploratory: check whether it marks the tissue boundary more cleanly than
sections 11-13.

In [ ]:
# ---- Calculation --------------------------------------------------------
# Per-FOV, per-z smoothed-Laplacian variance; reads raw frames, own per-FOV cache.
texture_stats_dir = cache_dir / "texture_stats" / f"round{target_round_id}"
texture_stats_dir.mkdir(parents=True, exist_ok=True)

fpath_by_fov   = {meta.fov_id_of_file(f): f for f in files}
frame_idx_list = [idx for idx, _ in z_frame_indices]   # shared by every per-frame reader below/after


def texture_stats_path(fpath):
    return texture_stats_dir / f"{Path(fpath).stem}_texture.npy"


texture_matrix     = np.full((len(fov_id_list), n_z), np.nan)
to_compute_texture = []
for i, fov_id in enumerate(fov_id_list):
    fpath = fpath_by_fov.get(fov_id)
    if fpath is None:
        continue
    if _npy_cache_valid(texture_stats_path(fpath)):
        texture_matrix[i, :] = np.load(texture_stats_path(fpath))
    else:
        to_compute_texture.append((i, fpath))

print(f"{len(fov_id_list) - len(to_compute_texture)} FOV texture profile(s) already cached; "
      f"{len(to_compute_texture)} to compute.")

# ---- Optional: submit a SLURM array job instead of computing locally -----
# USE_SLURM_ARRAY=True submits one array task per FOV still missing a cache
# file; re-run this cell after the job to load them.
USE_SLURM_ARRAY         = False   # set True on a cluster login node
SLURM_ARRAY_CONCURRENCY = 50      # max concurrently-running array tasks
SLURM_MEM               = "4gb"
SLURM_TIME              = "00:30:00"

if to_compute_texture and USE_SLURM_ARRAY:
    from MERci.acquisition.cluster_submit import (
        build_texture_stats_array_script, submit_sbatch, is_job_active,
    )

    texture_job_sentinel = cache_dir / f"texture_stats_job_round{target_round_id}.json"
    cached_job = json.loads(texture_job_sentinel.read_text()) if texture_job_sentinel.exists() else None

    if (cached_job is not None and cached_job.get("n_pending") == len(to_compute_texture)
            and is_job_active(cached_job["job_id"])):
        print(f"SLURM array job {cached_job['job_id']} is still active "
              f"({len(to_compute_texture)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"texture_stats_manifest_round{target_round_id}.txt"
        manifest_path.write_text("\n".join(str(fpath) for _, fpath in to_compute_texture) + "\n")

        script_path    = cache_dir / f"texture_stats_round{target_round_id}.sh"
        build_texture_stats_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=texture_stats_dir,
            frame_indices=frame_idx_list, n_pending=len(to_compute_texture), output_path=script_path,
            sigma=TEXTURE_SMOOTH_SIGMA, array_concurrency=SLURM_ARRAY_CONCURRENCY,
            mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            texture_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_compute_texture)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_compute_texture)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_compute_texture:
    reporter = ProgressReporter(total=len(to_compute_texture), label="Computing per-FOV texture profiles")
    for i, fpath in reporter.wrap(to_compute_texture):
        profile = np.full(n_z, np.nan)
        for pos, (_, frame) in enumerate(iter_image_frames(
            fpath, frame_idx_list, frame_width=config.frame_width, frame_height=config.frame_height,
        )):
            smoothed     = gaussian_filter(frame.astype(np.float64), sigma=TEXTURE_SMOOTH_SIGMA)
            profile[pos] = laplace(smoothed).var()
        atomic_save(texture_stats_path(fpath), lambda tmp: np.save(tmp, profile))
        texture_matrix[i, :] = profile

In [ ]:
# ---- Display --------------------------------------------------------------
order                 = np.argsort(texture_matrix[:, 0])
sorted_texture_matrix = texture_matrix[order]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for row in texture_matrix:
    axes[0].plot(z_axis_um, row, "-", color="steelblue", alpha=0.5, lw=1.0)
axes[0].set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_ylabel("Smoothed-Laplacian variance", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_title("Per-FOV texture profiles (overlaid)", fontsize=PLOT_TITLE_FONTSIZE)
axes[0].tick_params(labelsize=PLOT_TICK_FONTSIZE)

im = axes[1].imshow(
    sorted_texture_matrix, aspect="auto", cmap="viridis",
    extent=[z_axis_um.min(), z_axis_um.max(), len(order), 0],
)
axes[1].set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_ylabel("FOV rank (sorted by first-z value)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_title("Per-FOV texture profiles (heatmap, sorted)", fontsize=PLOT_TITLE_FONTSIZE)
axes[1].tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Smoothed-Laplacian variance", fontsize=PLOT_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- per-FOV texture (smoothed-Laplacian variance) profiles "
             f"({len(fov_id_list)} FOVs)", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

texture_profile_fig_path = figures_dir / f"tissue_thickness_texture_profile_round{target_round_id}.png"
fig.savefig(texture_profile_fig_path, dpi=150)
plt.show()
print(f"Saved: {texture_profile_fig_path}")

print("\nReview: does this statistic show a clean rise where real tissue signal starts (compare against "
      "the intensity/TPC views in sections 11-13, and the last-z mosaic in section 10)? If so, it could "
      "replace or complement THRESHOLD/TPC_THRESHOLD for deriving z_first_um/z_last_um in section 6.")

## 15 — Despike: remove single-z jumps from the texture profiles

A jump lasting one z-step is likely debris or a sensor artifact. A median
filter along z (`MEDIAN_FILTER_SIZE`, per FOV) removes it and keeps features
spanning at least `MEDIAN_FILTER_SIZE` z-steps.

In [ ]:
MEDIAN_FILTER_SIZE = 3   # z-steps; keep odd so the window is centered on each z

# Filter along z only (axis=1), never across FOVs.
texture_matrix_cleaned = median_filter(texture_matrix, size=(1, MEDIAN_FILTER_SIZE), mode="nearest")

order                         = np.argsort(texture_matrix_cleaned[:, 0])
sorted_texture_matrix_cleaned = texture_matrix_cleaned[order]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for row in texture_matrix_cleaned:
    axes[0].plot(z_axis_um, row, "-", color="steelblue", alpha=0.5, lw=1.0)
axes[0].set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_ylabel("Smoothed-Laplacian variance (despiked)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_title("Per-FOV texture profiles (despiked, overlaid)", fontsize=PLOT_TITLE_FONTSIZE)
axes[0].tick_params(labelsize=PLOT_TICK_FONTSIZE)

im = axes[1].imshow(
    sorted_texture_matrix_cleaned, aspect="auto", cmap="viridis",
    extent=[z_axis_um.min(), z_axis_um.max(), len(order), 0],
)
axes[1].set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_ylabel("FOV rank (sorted by first-z value)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_title("Per-FOV texture profiles (despiked, heatmap, sorted)", fontsize=PLOT_TITLE_FONTSIZE)
axes[1].tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Smoothed-Laplacian variance (despiked)", fontsize=PLOT_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- per-FOV texture profiles, despiked (median filter, "
             f"size={MEDIAN_FILTER_SIZE}) ({len(fov_id_list)} FOVs)", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

texture_cleaned_fig_path = figures_dir / f"tissue_thickness_texture_profile_despiked_round{target_round_id}.png"
fig.savefig(texture_cleaned_fig_path, dpi=150)
plt.show()
print(f"Saved: {texture_cleaned_fig_path}")

## 16 — Normalize texture profiles by per-FOV peak, guarded by a noise floor

Laplacian variance grows with how much of the field cells cover. Dividing each
despiked profile by its own peak makes shapes comparable across FOVs.

`TEXTURE_NOISE_FLOOR` is the highest despiked texture value among section 5's
background frames. FOVs whose peak is below it are left out (still visible in
section 15).

In [ ]:
# Despiked texture values of section 5's background frames (no new reads).
fov_row_by_id = {fov_id: i for i, fov_id in enumerate(fov_id_list)}
background_texture_values = [
    texture_matrix_cleaned[fov_row_by_id[int(fov_id)], int(pos)]
    for fov_id, pos in zip(worst_fov_ids, worst_pos)
    if int(fov_id) in fov_row_by_id
]
TEXTURE_NOISE_FLOOR = float(max(background_texture_values)) if background_texture_values else 0.0
print(f"TEXTURE_NOISE_FLOOR (from {len(background_texture_values)} confidently-background frame(s), "
      f"section 5): {TEXTURE_NOISE_FLOOR:.4f}")

fov_peak      = texture_matrix_cleaned.max(axis=1)
can_normalize = fov_peak > TEXTURE_NOISE_FLOOR
n_below_floor = int((~can_normalize).sum())
print(f"{n_below_floor} / {len(fov_id_list)} FOV(s) had a peak at or below TEXTURE_NOISE_FLOOR -- "
      f"no confident signal to normalize against; excluded from the normalized view below (still "
      f"visible in section 15's un-normalized, despiked view).")

# NaN for FOVs below the floor, so they are left out of the plot.
texture_matrix_normalized = np.where(
    can_normalize[:, None], texture_matrix_cleaned / fov_peak[:, None], np.nan,
)

order                            = np.argsort(texture_matrix_normalized[:, 0])
sorted_texture_matrix_normalized = texture_matrix_normalized[order]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for row in texture_matrix_normalized:
    if np.all(np.isnan(row)):
        continue
    axes[0].plot(z_axis_um, row, "-", color="steelblue", alpha=0.5, lw=1.0)
axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_ylabel("Texture (fraction of FOV's own peak)", fontsize=PLOT_LABEL_FONTSIZE)
axes[0].set_title(f"Per-FOV normalized profiles ({int(can_normalize.sum())}/{len(fov_id_list)} "
                   f"above noise floor)", fontsize=PLOT_TITLE_FONTSIZE)
axes[0].tick_params(labelsize=PLOT_TICK_FONTSIZE)

im = axes[1].imshow(
    sorted_texture_matrix_normalized, aspect="auto", cmap="viridis", vmin=0, vmax=1,
    extent=[z_axis_um.min(), z_axis_um.max(), len(order), 0],
)
axes[1].set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_ylabel("FOV rank (sorted by first-z value)", fontsize=PLOT_LABEL_FONTSIZE)
axes[1].set_title("Per-FOV normalized profiles (heatmap, sorted)", fontsize=PLOT_TITLE_FONTSIZE)
axes[1].tick_params(labelsize=PLOT_TICK_FONTSIZE)
cbar = fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Fraction of FOV's own peak", fontsize=PLOT_LABEL_FONTSIZE)
cbar.ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

fig.suptitle(f"Round {target_round_id} -- per-FOV texture profiles normalized by their own peak "
             f"({len(fov_id_list)} FOVs)", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

texture_normalized_fig_path = figures_dir / f"tissue_thickness_texture_profile_normalized_round{target_round_id}.png"
fig.savefig(texture_normalized_fig_path, dpi=150)
plt.show()
print(f"Saved: {texture_normalized_fig_path}")

## 17 — Texture-based z_last estimate (combined slope+value plateau detector)

`z_last_um_texture` from the despiked texture profile. Slope alone is fooled by
a shoulder mid-decay, so both slope and value (relative to the FOV's
peak-to-floor range) must stay low for `TEXTURE_RUN_LENGTH` z-steps. FOVs that
never plateau get `NaN`.

Section 18 renders a mosaic from these values, to compare with section 10.

In [ ]:
TEXTURE_REL_SLOPE_TOL = 0.01   # local slope, relative to the profile's own peak-to-floor range
TEXTURE_REL_VALUE_TOL = 0.15   # value itself, relative to the same range
TEXTURE_RUN_LENGTH    = 8      # consecutive z-steps both conditions must hold for


def detect_plateau_combined(profile, rel_slope_tol=TEXTURE_REL_SLOPE_TOL,
                             rel_value_tol=TEXTURE_REL_VALUE_TOL, run_length=TEXTURE_RUN_LENGTH):
    """First z-index (at/after the profile's own peak) where BOTH the local
    slope AND the value itself (both relative to the profile's own
    peak-minus-min range) stay below tolerance for a sustained run of
    run_length z-steps. Returns None if no such run exists -- e.g. a FOV
    whose real signal never actually plateaus within the imaged z-range, or
    a purely-flat/no-signal profile with no real peak to scan forward from.
    """
    n_local  = len(profile)
    peak_idx = int(np.argmax(profile))
    rng      = profile.max() - profile.min()
    if rng <= 0:
        return peak_idx
    slope     = np.abs(np.gradient(profile))
    rel_slope = slope / rng
    rel_value = (profile - profile.min()) / rng
    ok        = (rel_slope < rel_slope_tol) & (rel_value < rel_value_tol)
    for i in range(peak_idx, n_local - run_length + 1):
        if ok[i:i + run_length].all():
            return i
    return None


texture_z_last_idx = [detect_plateau_combined(row) for row in texture_matrix_cleaned]
texture_results_df = pd.DataFrame({"fov_id": fov_id_list, "z_last_idx_texture": texture_z_last_idx})
texture_results_df["z_last_um_texture"] = texture_results_df["z_last_idx_texture"].apply(
    lambda i: float(z_axis_um[int(i)]) if pd.notna(i) else np.nan
)

n_found = int(texture_results_df["z_last_um_texture"].notna().sum())
print(f"Texture-based plateau found for {n_found} / {len(texture_results_df)} FOV(s) "
      f"({len(texture_results_df) - n_found} declined -- no confident plateau within the imaged z-range).")

texture_results_csv = config.analysis_dir / f"tissue_thickness_texture_zlast_round{target_round_id}.csv"
texture_results_df.to_csv(texture_results_csv, index=False)
print(f"Saved: {texture_results_csv}")

## 18 — Verify: last-z mosaic using the texture-based z_last (section 17)

Section 10's mosaic at each FOV's `z_last_um_texture`, with a separate
thumbnail cache.

FOVs with no plateau render at the deepest z, with a white border and a `*`
label. Unmarked tiles should look like background; marked tiles should still
look like tissue. Compare with section 10.

In [ ]:
# ---- Calculation --------------------------------------------------------
last_z_raw_thumbnails_texture_dir = cache_dir / "last_z_raw_thumbnails_texture" / f"round{target_round_id}"
last_z_raw_thumbnails_texture_dir.mkdir(parents=True, exist_ok=True)


def last_z_raw_thumbnail_texture_path(fov_id):
    return last_z_raw_thumbnails_texture_dir / f"fov{fov_id:04d}.npy"


# FOVs with a plateau render at z_last_um_texture; the rest at the deepest z,
# which should still look like tissue.
full_depth_um          = float(z_axis_um[-1])
render_z_by_fov         = dict(zip(texture_results_df["fov_id"], texture_results_df["z_last_um_texture"].fillna(full_depth_um)))
full_thickness_fov_ids  = set(texture_results_df.loc[texture_results_df["z_last_um_texture"].isna(), "fov_id"])
fov_ids_all_texture     = texture_results_df["fov_id"].tolist()

to_render_texture = [f for f in fov_ids_all_texture if not _npy_cache_valid(last_z_raw_thumbnail_texture_path(f))]
print(f"{len(fov_ids_all_texture)} FOV(s) total ({len(full_thickness_fov_ids)} full-thickness, rendered "
      f"at the deepest available z instead of a texture-based plateau); "
      f"{len(fov_ids_all_texture) - len(to_render_texture)} thumbnail(s) already cached, "
      f"{len(to_render_texture)} to render.")

if to_render_texture:
    tw, th = config.thumbnail_size
    reporter = ProgressReporter(total=len(to_render_texture), label="Rendering texture-based last-z raw thumbnails")
    for fov_id in reporter.wrap(to_render_texture):
        z_last    = float(render_z_by_fov[fov_id])
        counters  = channel_counters[fov_id]
        pos       = int(np.argmin(np.abs(counters["z_um"] - z_last)))
        frame_idx = int(counters["frame_indices"][pos])

        fpath = fpath_by_fov.get(fov_id)
        frame = next(frame for _, frame in iter_image_frames(
            fpath, [frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
        ))
        frame = apply_microscope_orientation(frame, **MICROSCOPE_ORIENTATION)
        thumb = sk_resize(frame.astype(np.float64), (th, tw), anti_aliasing=True, preserve_range=True)
        atomic_save(last_z_raw_thumbnail_texture_path(fov_id), lambda tmp: np.save(tmp, thumb.astype(np.float32)))

In [ ]:
# ---- Display --------------------------------------------------------------
raw_thumbnails_texture = {fov_id: np.load(last_z_raw_thumbnail_texture_path(fov_id))
                           for fov_id in fov_ids_all_texture}

pooled_pixels_texture = np.concatenate([t.ravel() for t in raw_thumbnails_texture.values()])
lo_pct, hi_pct        = config.thumbnail_percentile_clip
vmin_t, vmax_t        = np.percentile(pooled_pixels_texture, [lo_pct, hi_pct])
print(f"Shared display scale (p{lo_pct:.0f}-p{hi_pct:.0f} over all {len(raw_thumbnails_texture)} FOV thumbnails): "
      f"[{vmin_t:.0f}, {vmax_t:.0f}]")

last_z_thumbnails_texture_uint8 = {fov_id: _to_uint8(t, vmin_t, vmax_t) for fov_id, t in raw_thumbnails_texture.items()}
last_z_positions_texture        = {fov_id: meta.fovs[fov_id].position for fov_id in fov_ids_all_texture}
# "*" suffix marks the full-thickness tiles' label as "deepest available z",
# distinct from a real detected z_last_um_texture value.
last_z_labels_texture = {
    fov_id: f"{render_z_by_fov[fov_id]:.0f}" + ("*" if fov_id in full_thickness_fov_ids else "")
    for fov_id in fov_ids_all_texture
}

# Red squares are a matplotlib overlay, not burned into the saved mosaic;
# return_tile_bboxes gives each tile's box.
last_z_mosaic_texture_path = figures_dir / f"tissue_thickness_last_z_mosaic_texture_round{target_round_id}.png"
canvas_texture, tile_bboxes_texture = create_mosaic(
    last_z_thumbnails_texture_uint8, last_z_positions_texture, last_z_mosaic_texture_path,
    thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding,
    flip_y=last_z_mosaic_flip_y, labels=last_z_labels_texture,
    return_tile_bboxes=True,
)

# 1 image pixel = 1 figure pixel, so the overlay lines up and the PNG isn't resampled.
overlay_dpi = 100
fig = plt.figure(figsize=(canvas_texture.shape[1] / overlay_dpi, canvas_texture.shape[0] / overlay_dpi),
                  dpi=overlay_dpi)
ax = fig.add_axes([0, 0, 1, 1])   # fill the whole figure, no margins
ax.imshow(canvas_texture, cmap="gray", interpolation="nearest", vmin=0, vmax=255)
ax.axis("off")
for fov_id in full_thickness_fov_ids:
    x0, y0, x1, y1 = tile_bboxes_texture[fov_id]
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="red", linewidth=1.2))

last_z_mosaic_texture_overlay_path = (
    figures_dir / f"tissue_thickness_last_z_mosaic_texture_round{target_round_id}_highlighted.png"
)
fig.savefig(last_z_mosaic_texture_overlay_path, dpi=overlay_dpi)
plt.show()
print(f"Saved: {last_z_mosaic_texture_overlay_path}")

print(f"\n{len(full_thickness_fov_ids)} FOV(s) had no texture-based plateau -- outlined in red above "
      f"(label suffixed '*'), rendered at the deepest available z instead of z_last_um_texture. Those "
      f"tiles should still look like real tissue, NOT background -- if one looks like clean background "
      f"instead, that FOV's classification is likely wrong (should have been a transition, not "
      f"full-thickness). Every other (unmarked) tile is rendered at its own z_last_um_texture and should "
      f"look like background/noise if that estimate is correct.")

## 19 — Max projection + measurement overlay for representative example FOVs

For a few FOVs: a max projection across X (side view of tissue vs. depth),
and below it, on the same z axis, the scaled mean intensity and raw Laplacian
variance.

`EXAMPLE_FOVS`: two with no signal, two that never plateau, two with a clean
tissue-to-background transition. Override to inspect others.

In [ ]:
# ---- Calculation --------------------------------------------------------
EXAMPLE_FOVS = {
    68:  "noise",
    618: "noise",
    158: "full_thickness",
    888: "full_thickness",
    458: "transition",
    396: "transition",
}

example_cache_dir = cache_dir / "example_overlay" / f"round{target_round_id}"
example_cache_dir.mkdir(parents=True, exist_ok=True)


def example_cache_path(fov_id):
    return example_cache_dir / f"fov{fov_id:04d}.npz"


example_data = {}
to_read = [fid for fid in EXAMPLE_FOVS if not _npy_cache_valid(example_cache_path(fid))]
for fid in EXAMPLE_FOVS:
    if fid not in to_read:
        cached = np.load(example_cache_path(fid))
        example_data[fid] = {k: cached[k] for k in cached.files}
print(f"{len(EXAMPLE_FOVS) - len(to_read)} example FOV(s) already cached; {len(to_read)} to read.")

if to_read:
    reporter = ProgressReporter(total=len(to_read), label="Reading example FOV z-stacks")
    for fov_id in reporter.wrap(to_read):
        fpath = fpath_by_fov.get(fov_id)
        if fpath is None:
            print(f"WARNING: FOV {fov_id} not found in this round -- skipping.")
            continue
        max_proj_rows  = []
        mean_intensity = np.full(n_z, np.nan)
        laplacian_var  = np.full(n_z, np.nan)
        for pos, (_, frame) in enumerate(iter_image_frames(
            fpath, frame_idx_list, frame_width=config.frame_width, frame_height=config.frame_height,
        )):
            max_proj_rows.append(frame.max(axis=1))   # collapse X -> one value per row (Y)
            mean_intensity[pos] = frame.mean()
            smoothed           = gaussian_filter(frame.astype(np.float64), sigma=TEXTURE_SMOOTH_SIGMA)
            laplacian_var[pos] = laplace(smoothed).var()
        result = {
            "max_proj":       np.stack(max_proj_rows, axis=1),   # (H, n_z) -- z along columns
            "mean_intensity": mean_intensity,
            "laplacian_var":  laplacian_var,
        }
        atomic_save(example_cache_path(fov_id), lambda tmp: np.savez_compressed(tmp, **result))
        example_data[fov_id] = result
print(f"{len(example_data)} example FOV(s) ready.")

In [ ]:
# ---- Display --------------------------------------------------------------
present = [fid for fid in EXAMPLE_FOVS if fid in example_data]
fig, axes = plt.subplots(2, len(present), figsize=(4 * len(present), 7), sharex="col", squeeze=False)

for col, fov_id in enumerate(present):
    label = EXAMPLE_FOVS[fov_id]
    d     = example_data[fov_id]
    ax_top, ax_bot = axes[0, col], axes[1, col]

    ax_top.imshow(d["max_proj"], aspect="auto", cmap="gray",
                  extent=[z_axis_um.min(), z_axis_um.max(), d["max_proj"].shape[0], 0])
    ax_top.set_title(f"FOV {fov_id} ({label})\nmax projection across X",
                      fontsize=max(PLOT_TITLE_FONTSIZE - 2, 8))
    ax_top.set_ylabel("Y (px)", fontsize=PLOT_LABEL_FONTSIZE)
    ax_top.tick_params(labelsize=PLOT_TICK_FONTSIZE)

    mean_scaled = d["mean_intensity"] / d["mean_intensity"].max()
    lap_scaled  = d["laplacian_var"] / d["laplacian_var"].max()
    ax_bot.plot(z_axis_um, mean_scaled, "-", color="steelblue", lw=1.5, label="mean intensity (scaled)")
    ax_bot.plot(z_axis_um, lap_scaled, "-", color="darkorange", lw=1.5, label="Laplacian variance (scaled)")
    ax_bot.set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax_bot.set_ylabel("scaled value", fontsize=PLOT_LABEL_FONTSIZE)
    ax_bot.tick_params(labelsize=PLOT_TICK_FONTSIZE)
    ax_bot.legend(fontsize=max(PLOT_LEGEND_FONTSIZE - 1, 7))

fig.suptitle(f"Round {target_round_id} -- max projection vs. measurement overlay, example FOVs",
             fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

example_overlay_fig_path = figures_dir / f"tissue_thickness_example_overlay_round{target_round_id}.png"
fig.savefig(example_overlay_fig_path, dpi=150)
plt.show()
print(f"Saved: {example_overlay_fig_path}")

## 20 — Tiers vs. ideal: how many z-depth tiers actually capture the available savings?

multi_z images bits rounds in `N_TIERS` z-depth tiers
(`04_create_hal_config_and_shutters_multi_z.ipynb`); each FOV is imaged to its
tier's max depth. Compares the ideal (one tier per FOV) with candidate tier
counts, using `z_last_um_texture` + `Z_MARGIN_UM` as each FOV's need (full
depth if no plateau), in frames, time and space.

Use it to pick `N_TIERS` in multi_z notebook 04.

In [ ]:
# ---- Calculation --------------------------------------------------------
TIER_CANDIDATE_COUNTS = [3, 5, 10, 15, 20]   # candidate N_TIERS values to compare against the ideal

# Per-FOV needed depth (same convention as sections 9/17): texture-based
# z_last + margin, capped at full depth; a FOV with no detected plateau gets
# the full depth (conservative default, same as section 17).
tier_full_depth_idx = n_z - 1
tier_margin_steps   = int(np.ceil(Z_MARGIN_UM / (z_axis_um[1] - z_axis_um[0])))
tier_needed_idx = np.where(
    texture_results_df["z_last_idx_texture"].isna(),
    tier_full_depth_idx,
    np.minimum(texture_results_df["z_last_idx_texture"].fillna(0) + tier_margin_steps, tier_full_depth_idx),
).astype(int)


def tier_assign(needed, n_tiers):
    """Quantile-bucket FOVs by needed depth into n_tiers groups; each FOV's
    assigned depth = the MAX needed depth within its own bucket -- same
    convention as multi_z's 04_create_hal_config_and_shutters_multi_z."""
    order = np.argsort(needed)
    bucket_of_rank = (np.arange(len(needed)) * n_tiers) // len(needed)
    assigned = np.empty_like(needed)
    for b in range(n_tiers):
        idx_in_bucket = order[bucket_of_rank == b]
        assigned[idx_in_bucket] = needed[idx_in_bucket].max()
    return assigned


baseline_total_idx = len(tier_needed_idx) * tier_full_depth_idx
ideal_total_idx     = int(tier_needed_idx.sum())

tier_comparison_rows = [{
    "scheme":               "ideal (per-FOV, no tiers)",
    "n_distinct_depths":    len(np.unique(tier_needed_idx)),
    "total_steps":          ideal_total_idx,
    "pct_saved_vs_baseline": 100 * (1 - ideal_total_idx / baseline_total_idx),
    "pct_more_than_ideal":  0.0,
}]
for n_tiers in TIER_CANDIDATE_COUNTS:
    assigned = tier_assign(tier_needed_idx, n_tiers)
    total    = int(assigned.sum())
    tier_comparison_rows.append({
        "scheme":               f"{n_tiers} tiers",
        "n_distinct_depths":    len(np.unique(assigned)),
        "total_steps":          total,
        "pct_saved_vs_baseline": 100 * (1 - total / baseline_total_idx),
        "pct_more_than_ideal":  100 * (total - ideal_total_idx) / ideal_total_idx,
    })

tier_comparison_df = pd.DataFrame(tier_comparison_rows)

# Convert frame steps to time/space with section 8's rate and section 9's
# frame size.
tier_comparison_df["bytes_saved_vs_baseline"] = (
    (baseline_total_idx - tier_comparison_df["total_steps"]) * frame_bytes
)
tier_comparison_df["time_saved_experimental_s"] = (
    (baseline_total_idx - tier_comparison_df["total_steps"]) * experimental_time_per_frame_s
)

tier_comparison_csv = config.analysis_dir / f"tissue_thickness_tier_comparison_round{target_round_id}.csv"
tier_comparison_df.to_csv(tier_comparison_csv, index=False)
print(tier_comparison_df.to_string(index=False))
print(f"\nSaved: {tier_comparison_csv}")

In [ ]:
# ---- Display --------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(tier_comparison_df))
ax.bar(x, tier_comparison_df["pct_saved_vs_baseline"], color="steelblue")
ax.set_xticks(x)
ax.set_xticklabels(tier_comparison_df["scheme"], rotation=30, ha="right", fontsize=PLOT_TICK_FONTSIZE)
ax.set_ylabel("% saved vs. full-depth baseline", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Round {target_round_id} -- tier count vs. achievable savings", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
for xi, pct in zip(x, tier_comparison_df["pct_saved_vs_baseline"]):
    ax.text(xi, pct + 0.5, f"{pct:.1f}%", ha="center", fontsize=PLOT_TICK_FONTSIZE)
fig.tight_layout()

tier_comparison_fig_path = figures_dir / f"tissue_thickness_tier_comparison_round{target_round_id}.png"
fig.savefig(tier_comparison_fig_path, dpi=150)
plt.show()
print(f"Saved: {tier_comparison_fig_path}")

print(f"\nTotal current (baseline, full depth): {format_bytes(baseline_total_idx * frame_bytes)}")
for _, row in tier_comparison_df.iterrows():
    extra = "(ideal)" if row["pct_more_than_ideal"] == 0 else f"({row['pct_more_than_ideal']:.1f}% more imaging than ideal)"
    print(f"  {row['scheme']:28s}: {format_bytes(row['bytes_saved_vs_baseline']):>12s} saved "
          f"({row['pct_saved_vs_baseline']:5.1f}%), {format_duration(row['time_saved_experimental_s'])} saved {extra}")

## 21 — Compare z_final per FOV: TPC-based (6/10) vs. texture-based (17/18)

Both `z_last` estimates per FOV (x = FOV id, y = `z_final`), to check for a
systematic difference, e.g. the texture estimate stopping shallower.

In [ ]:
z_compare_df = results_df[["fov_id", "z_last_um"]].merge(
    texture_results_df[["fov_id", "z_last_um_texture"]], on="fov_id", how="outer"
).sort_values("fov_id")

fig, ax = plt.subplots(figsize=(max(10, len(z_compare_df) * 0.02), 5))
ax.plot(z_compare_df["fov_id"], z_compare_df["z_last_um"], "o", color="steelblue", alpha=0.5, ms=3,
        label="TPC-based (section 6/10)")
ax.plot(z_compare_df["fov_id"], z_compare_df["z_last_um_texture"], "o", color="darkorange", alpha=0.5, ms=3,
        label="Texture-based (section 17/18)")
ax.set_xlabel("FOV id", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("z_final (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Round {target_round_id} -- z_final per FOV: TPC-based vs. texture-based",
             fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
ax.legend(fontsize=PLOT_LEGEND_FONTSIZE)
fig.tight_layout()

z_compare_fig_path = figures_dir / f"tissue_thickness_zfinal_compare_round{target_round_id}.png"
fig.savefig(z_compare_fig_path, dpi=150)
plt.show()
print(f"Saved: {z_compare_fig_path}")

# Quantify the systematic difference directly, not just visually.
both_estimates = z_compare_df.dropna(subset=["z_last_um", "z_last_um_texture"])
diff = both_estimates["z_last_um_texture"] - both_estimates["z_last_um"]
print(f"\n{len(both_estimates)} / {len(z_compare_df)} FOV(s) have both estimates "
      f"({len(z_compare_df) - len(both_estimates)} missing one or both -- TPC found no signal, "
      f"or the texture detector found no plateau).")
print(f"texture - TPC difference: mean={diff.mean():.2f} um, median={diff.median():.2f} um, "
      f"std={diff.std():.2f} um")
print(f"Texture-based is LOWER (shallower) than TPC-based for {int((diff < 0).sum())}/{len(both_estimates)} "
      f"FOV(s) ({100*(diff < 0).mean():.1f}%).")

## 22 — Diagnose the TPC-vs-texture disagreement: worst-offender profiles

The FOVs with the largest disagreement: despiked texture profile with
`z_last_um` (TPC, blue) and `z_last_um_texture` (orange). Shows where the
texture criterion fires early, e.g. on a sparse tail of signal that TPC still
counts.

In [ ]:
# Largest (texture - TPC) disagreements, most negative (texture much shallower) first.
diagnostic_df = z_compare_df.dropna(subset=["z_last_um", "z_last_um_texture"]).copy()
diagnostic_df["diff_um"] = diagnostic_df["z_last_um_texture"] - diagnostic_df["z_last_um"]
worst = diagnostic_df.sort_values("diff_um").head(6)
print(worst.to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, row in zip(axes.ravel(), worst.itertuples()):
    fov_id = row.fov_id
    i = fov_id_list.index(fov_id) if fov_id in fov_id_list else None
    if i is None:
        ax.set_title(f"FOV {fov_id}: not in texture_matrix_cleaned")
        continue
    ax.plot(z_axis_um, texture_matrix_cleaned[i], "-", color="steelblue", lw=1.3, label="despiked texture")
    ax.axvline(row.z_last_um, color="steelblue", linestyle="--", lw=1.5,
               label=f"TPC z_last={row.z_last_um:.1f} um")
    ax.axvline(row.z_last_um_texture, color="darkorange", linestyle="--", lw=1.5,
               label=f"texture z_last={row.z_last_um_texture:.1f} um")
    ax.set_title(f"FOV {fov_id} (diff={row.diff_um:.1f} um)", fontsize=PLOT_TITLE_FONTSIZE - 2)
    ax.set_xlabel("z (um)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.set_ylabel("texture (despiked)", fontsize=PLOT_LABEL_FONTSIZE)
    ax.legend(fontsize=7)
fig.suptitle(f"Round {target_round_id} -- worst TPC-vs-texture disagreements", fontsize=PLOT_SUPTITLE_FONTSIZE)
fig.tight_layout()

disagreement_fig_path = figures_dir / f"tissue_thickness_zlast_disagreement_round{target_round_id}.png"
fig.savefig(disagreement_fig_path, dpi=150)
plt.show()
print(f"Saved: {disagreement_fig_path}")

### 22b — Does the disagreement correlate with each FOV's own brightness?

Shot noise grows with signal, but `detect_plateau_combined` (17) normalizes
only within each FOV's peak-to-floor range. If that drives the disagreement,
`diff_um` (texture - TPC) should trend with each FOV's peak intensity.

In [ ]:
# Per-FOV peak intensity (mean of its brightest frame), from the Counters.
fov_peak_intensity = {}
for fov_id in fov_id_list:
    counters = channel_counters[fov_id]
    means = [counter_mean(v, c) for v, c in zip(counters["values_per_z"], counters["counts_per_z"])]
    fov_peak_intensity[fov_id] = max(means)

diagnostic_df["peak_intensity"] = diagnostic_df["fov_id"].map(fov_peak_intensity)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(diagnostic_df["peak_intensity"], diagnostic_df["diff_um"], s=10, alpha=0.4, color="steelblue")
ax.axhline(0, color="gray", linestyle=":", lw=1)
ax.set_xlabel("FOV peak intensity (mean of brightest frame)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("texture z_last - TPC z_last (um)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title(f"Round {target_round_id} -- disagreement vs. FOV brightness", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)
fig.tight_layout()

intensity_corr_fig_path = figures_dir / f"tissue_thickness_zlast_diff_vs_intensity_round{target_round_id}.png"
fig.savefig(intensity_corr_fig_path, dpi=150)
plt.show()
print(f"Saved: {intensity_corr_fig_path}")

corr = diagnostic_df[["peak_intensity", "diff_um"]].corr().iloc[0, 1]
print(f"\nCorrelation (diff_um vs. peak_intensity): {corr:.3f}")
print("Lowest-intensity FOVs' diff_um:")
print(diagnostic_df.nsmallest(5, "peak_intensity")[["fov_id", "peak_intensity", "diff_um"]].to_string(index=False))
print("\nHighest-intensity FOVs' diff_um:")
print(diagnostic_df.nlargest(5, "peak_intensity")[["fov_id", "peak_intensity", "diff_um"]].to_string(index=False))

## 23 — Simple fallback: TPC-based z_last + a small fixed margin (1-10 um)

Keeps the TPC `z_last_um` and adds a fixed margin (1-10 um), one mosaic per
margin. If a small margin already gives clean background tiles, use that.

Each FOV is read once, over a window from `z_last_um` to the largest margin.

In [ ]:
# ---- Calculation --------------------------------------------------------
MARGIN_CANDIDATES_UM = list(range(1, 11))   # 1 to 10 um, simple additive correction to TPC-based z_last

z_step_um         = float(z_axis_um[1] - z_axis_um[0])
max_margin_steps  = int(np.ceil(max(MARGIN_CANDIDATES_UM) / z_step_um))
full_depth_um_tpc = float(z_axis_um[-1])

fov_ids_with_tpc_signal = results_df.loc[results_df["z_last_um"].notna(), "fov_id"].tolist()

margin_thumbnails_dir = cache_dir / "tpc_margin_sweep" / f"round{target_round_id}"
margin_thumbnails_dir.mkdir(parents=True, exist_ok=True)


def margin_thumbnail_path(fov_id, margin_um):
    return margin_thumbnails_dir / f"fov{fov_id:04d}_margin{margin_um}.npy"


to_read = [
    fov_id for fov_id in fov_ids_with_tpc_signal
    if not all(_npy_cache_valid(margin_thumbnail_path(fov_id, m)) for m in MARGIN_CANDIDATES_UM)
]
print(f"{len(fov_ids_with_tpc_signal)} FOV(s) with TPC-based signal; "
      f"{len(fov_ids_with_tpc_signal) - len(to_read)} fully cached across all {len(MARGIN_CANDIDATES_UM)} "
      f"margins already; {len(to_read)} FOV(s) need at least one margin computed.")

# ---- Optional: submit a SLURM array job instead of computing locally -----
# USE_SLURM_ARRAY=True submits one array task per FOV still missing a cache
# file; re-run this cell after the job to load them.
USE_SLURM_ARRAY         = False   # set True on a cluster login node
SLURM_ARRAY_CONCURRENCY = 50
SLURM_MEM               = "4gb"
SLURM_TIME              = "00:15:00"

if to_read and USE_SLURM_ARRAY:
    from MERci.acquisition.cluster_submit import (
        build_tpc_margin_array_script, submit_sbatch, is_job_active,
    )
    import csv

    margin_job_sentinel = cache_dir / f"tpc_margin_job_round{target_round_id}.json"
    cached_margin_job = json.loads(margin_job_sentinel.read_text()) if margin_job_sentinel.exists() else None

    if (cached_margin_job is not None and cached_margin_job.get("n_pending") == len(to_read)
            and is_job_active(cached_margin_job["job_id"])):
        print(f"SLURM array job {cached_margin_job['job_id']} is still active "
              f"({len(to_read)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"tpc_margin_manifest_round{target_round_id}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path", "z_last_um"])
            for fov_id in to_read:
                z_last_um_tpc = float(results_df.loc[results_df["fov_id"] == fov_id, "z_last_um"].iloc[0])
                writer.writerow([fov_id, fpath_by_fov[fov_id], z_last_um_tpc])

        script_path = cache_dir / f"tpc_margin_round{target_round_id}.sh"
        build_tpc_margin_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=margin_thumbnails_dir,
            frame_indices=frame_idx_list, z_um_values=z_axis_um.tolist(), margins=MARGIN_CANDIDATES_UM,
            thumbnail_size=config.thumbnail_size, orientation=MICROSCOPE_ORIENTATION,
            n_pending=len(to_read), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY, mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            margin_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(to_read)}))
            print(f"Submitted SLURM array job {job_id} for {len(to_read)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_read:
    tw, th = config.thumbnail_size
    reporter = ProgressReporter(total=len(to_read), label="Rendering TPC+margin thumbnails")
    for fov_id in reporter.wrap(to_read):
        z_last_um_tpc = float(results_df.loc[results_df["fov_id"] == fov_id, "z_last_um"].iloc[0])
        counters      = channel_counters[fov_id]

        # One window read covers every margin candidate.
        pos_start        = int(np.argmin(np.abs(counters["z_um"] - z_last_um_tpc)))
        pos_end          = min(pos_start + max_margin_steps, n_z - 1)
        window_positions = list(range(pos_start, pos_end + 1))
        window_z_um      = counters["z_um"][pos_start:pos_end + 1]
        window_frame_idx = [int(counters["frame_indices"][p]) for p in window_positions]

        fpath = fpath_by_fov.get(fov_id)
        frames_by_pos = {}
        for pos, (_, frame) in zip(window_positions, iter_image_frames(
            fpath, window_frame_idx, frame_width=config.frame_width, frame_height=config.frame_height,
        )):
            frame = apply_microscope_orientation(frame, **MICROSCOPE_ORIENTATION)
            frames_by_pos[pos] = sk_resize(
                frame.astype(np.float64), (th, tw), anti_aliasing=True, preserve_range=True
            ).astype(np.float32)

        for margin_um in MARGIN_CANDIDATES_UM:
            z_target  = min(z_last_um_tpc + margin_um, full_depth_um_tpc)
            local_idx = int(np.argmin(np.abs(window_z_um - z_target)))
            pos       = pos_start + local_idx
            atomic_save(margin_thumbnail_path(fov_id, margin_um), lambda tmp: np.save(tmp, frames_by_pos[pos]))

In [ ]:
# ---- Display --------------------------------------------------------------
# One shared intensity scale across every margin candidate (not one per
# mosaic), so brightness is directly comparable margin-to-margin.
margin_thumbnails_by_margin = {
    margin_um: {fov_id: np.load(margin_thumbnail_path(fov_id, margin_um)) for fov_id in fov_ids_with_tpc_signal}
    for margin_um in MARGIN_CANDIDATES_UM
}
pooled_pixels_margin = np.concatenate([
    t.ravel() for thumbs in margin_thumbnails_by_margin.values() for t in thumbs.values()
])
lo_pct, hi_pct       = config.thumbnail_percentile_clip
vmin_m, vmax_m       = np.percentile(pooled_pixels_margin, [lo_pct, hi_pct])
print(f"Shared display scale across all {len(MARGIN_CANDIDATES_UM)} margins "
      f"(p{lo_pct:.0f}-p{hi_pct:.0f}): [{vmin_m:.0f}, {vmax_m:.0f}]")

margin_positions = {fov_id: meta.fovs[fov_id].position for fov_id in fov_ids_with_tpc_signal}

for margin_um in MARGIN_CANDIDATES_UM:
    thumbs_uint8 = {fov_id: _to_uint8(t, vmin_m, vmax_m)
                    for fov_id, t in margin_thumbnails_by_margin[margin_um].items()}
    margin_mosaic_path = figures_dir / f"tissue_thickness_tpc_margin{margin_um}_mosaic_round{target_round_id}.png"
    create_mosaic(thumbs_uint8, margin_positions, margin_mosaic_path,
                  thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding,
                  flip_y=last_z_mosaic_flip_y)
    print(f"\n--- margin = +{margin_um} um ---")
    display_mosaic(margin_mosaic_path, target_round_id)

print(f"\nReview each margin above: pick the smallest one where tiles consistently look like clean "
      f"background, not still-mid-tissue. That value + TPC z_last_um is the simple fallback depth.")

## 24 — Animated GIF: tissue disappearing across the full z-range

One mosaic of every FOV per z position, as an animated GIF.

**Slow**: reads every FOV at every included z-step. `GIF_Z_STRIDE` subsamples
z, and the cell prints a time estimate before reading. Thumbnails are cached
per (z, FOV). `USE_SLURM_ARRAY` submits one array task per FOV
(`build_gif_frames_array_script` + `cli_compute_gif_frame_thumbnails.py`).

In [ ]:
# ---- Calculation --------------------------------------------------------
GIF_Z_STRIDE           = 5      # every Nth z-step (of n_z total) -- lower = smoother GIF, more reads
GIF_DOWNSCALE_WIDTH_PX  = 1000   # each mosaic frame is downscaled to this width before going into the GIF
GIF_FRAME_DURATION_MS   = 300    # per-frame display duration in the saved GIF

gif_z_positions = list(range(0, n_z, GIF_Z_STRIDE))
n_gif_reads      = len(gif_z_positions) * len(fov_id_list)
est_seconds      = n_gif_reads * experimental_time_per_frame_s
print(f"GIF_Z_STRIDE={GIF_Z_STRIDE} -> {len(gif_z_positions)} z-step(s) x {len(fov_id_list)} FOV(s) "
      f"= {n_gif_reads} frame read(s), estimated {format_duration(est_seconds)} at this round's measured "
      f"experimental rate ({experimental_time_per_frame_s:.4f} s/frame, section 8) -- lower GIF_Z_STRIDE "
      f"for a smoother animation (more reads, more time), raise it for a quicker rough preview.")

gif_frames_dir = cache_dir / "gif_frames" / f"round{target_round_id}"
gif_frames_dir.mkdir(parents=True, exist_ok=True)


def gif_frame_thumbnail_path(z_pos, fov_id):
    return gif_frames_dir / f"z{z_pos:04d}_fov{fov_id:04d}.npy"


to_read = [
    (z_pos, fov_id) for z_pos in gif_z_positions for fov_id in fov_id_list
    if not _npy_cache_valid(gif_frame_thumbnail_path(z_pos, fov_id))
]
fovs_needing_read = sorted({fov_id for _, fov_id in to_read})
print(f"{n_gif_reads - len(to_read)} thumbnail(s) already cached; {len(to_read)} to read "
      f"({len(fovs_needing_read)} FOV(s) affected).")

# ---- Optional: submit a SLURM array job instead of computing locally -----
# USE_SLURM_ARRAY=True submits one array task per FOV still missing a cache
# file; re-run this cell after the job to load them.
USE_SLURM_ARRAY         = False   # set True on a cluster login node
SLURM_ARRAY_CONCURRENCY = 50
SLURM_MEM               = "4gb"
SLURM_TIME              = "00:20:00"

if fovs_needing_read and USE_SLURM_ARRAY:
    from MERci.acquisition.cluster_submit import (
        build_gif_frames_array_script, submit_sbatch, is_job_active,
    )
    import csv

    gif_job_sentinel = cache_dir / f"gif_frames_job_round{target_round_id}.json"
    cached_gif_job = json.loads(gif_job_sentinel.read_text()) if gif_job_sentinel.exists() else None

    if (cached_gif_job is not None and cached_gif_job.get("n_pending") == len(fovs_needing_read)
            and is_job_active(cached_gif_job["job_id"])):
        print(f"SLURM array job {cached_gif_job['job_id']} is still active "
              f"({len(fovs_needing_read)} FOV(s) pending) -- re-run this cell later once it finishes.")
    else:
        manifest_path = cache_dir / f"gif_frames_manifest_round{target_round_id}.csv"
        with open(manifest_path, "w", newline="") as fh:
            writer = csv.writer(fh)
            writer.writerow(["fov_id", "image_path"])
            for fov_id in fovs_needing_read:
                writer.writerow([fov_id, fpath_by_fov[fov_id]])

        script_path = cache_dir / f"gif_frames_round{target_round_id}.sh"
        build_gif_frames_array_script(
            sample_dir=SAMPLE_DIR, manifest_path=manifest_path, output_dir=gif_frames_dir,
            z_positions=gif_z_positions, frame_indices=frame_idx_list,
            thumbnail_size=config.thumbnail_size, orientation=MICROSCOPE_ORIENTATION,
            n_pending=len(fovs_needing_read), output_path=script_path,
            array_concurrency=SLURM_ARRAY_CONCURRENCY, mem=SLURM_MEM, time=SLURM_TIME,
        )
        job_id = submit_sbatch(script_path)
        if job_id is not None:
            gif_job_sentinel.write_text(json.dumps({"job_id": job_id, "n_pending": len(fovs_needing_read)}))
            print(f"Submitted SLURM array job {job_id} for {len(fovs_needing_read)} FOV(s) -- "
                  f"re-run this cell later once it finishes to load the results.")
        else:
            print("sbatch submission failed (see the logged error above) -- fix the issue and re-run this cell.")
elif to_read:
    tw, th = config.thumbnail_size
    reporter = ProgressReporter(total=len(to_read), label="Rendering GIF frame thumbnails")
    for z_pos, fov_id in reporter.wrap(to_read):
        counters  = channel_counters[fov_id]
        frame_idx = int(counters["frame_indices"][z_pos])
        fpath     = fpath_by_fov.get(fov_id)
        if fpath is None:
            continue
        frame = next(frame for _, frame in iter_image_frames(
            fpath, [frame_idx], frame_width=config.frame_width, frame_height=config.frame_height,
        ))
        frame = apply_microscope_orientation(frame, **MICROSCOPE_ORIENTATION)
        thumb = sk_resize(frame.astype(np.float64), (th, tw), anti_aliasing=True, preserve_range=True)
        atomic_save(gif_frame_thumbnail_path(z_pos, fov_id), lambda tmp: np.save(tmp, thumb.astype(np.float32)))

In [ ]:
# ---- Display --------------------------------------------------------------
gif_positions = {fov_id: meta.fovs[fov_id].position for fov_id in fov_id_list}

# One shared intensity scale across z-steps, so fading is real signal.
pooled_pixels_gif = np.concatenate([
    np.load(gif_frame_thumbnail_path(z_pos, fov_id)).ravel()
    for z_pos in gif_z_positions for fov_id in fov_id_list
    if _npy_cache_valid(gif_frame_thumbnail_path(z_pos, fov_id))
])
lo_pct, hi_pct = config.thumbnail_percentile_clip
vmin_g, vmax_g = np.percentile(pooled_pixels_gif, [lo_pct, hi_pct])
print(f"Shared GIF display scale (p{lo_pct:.0f}-p{hi_pct:.0f}): [{vmin_g:.0f}, {vmax_g:.0f}]")

gif_frames_dir_png = cache_dir / "gif_frames_png" / f"round{target_round_id}"
gif_frames_dir_png.mkdir(parents=True, exist_ok=True)

pil_frames = []
reporter = ProgressReporter(total=len(gif_z_positions), label="Assembling GIF frames")
for z_pos in reporter.wrap(gif_z_positions):
    thumbs_uint8 = {
        fov_id: _to_uint8(np.load(gif_frame_thumbnail_path(z_pos, fov_id)), vmin_g, vmax_g)
        for fov_id in fov_id_list if _npy_cache_valid(gif_frame_thumbnail_path(z_pos, fov_id))
    }
    frame_png_path = gif_frames_dir_png / f"z{z_pos:04d}.png"
    create_mosaic(thumbs_uint8, gif_positions, frame_png_path,
                  thumbnail_size=config.thumbnail_size, padding=config.mosaic_padding,
                  flip_y=last_z_mosaic_flip_y)

    img = Image.open(frame_png_path)
    scale = GIF_DOWNSCALE_WIDTH_PX / img.width
    img = img.resize((GIF_DOWNSCALE_WIDTH_PX, max(1, int(img.height * scale))))
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=max(16, GIF_DOWNSCALE_WIDTH_PX // 40))
    z_um = float(z_axis_um[z_pos])
    draw.text((10, 10), f"z = {z_um:.1f} um", fill=255, font=font)
    pil_frames.append(img)

gif_path = figures_dir / f"tissue_thickness_z_sweep_round{target_round_id}.gif"
pil_frames[0].save(
    gif_path, save_all=True, append_images=pil_frames[1:],
    duration=GIF_FRAME_DURATION_MS, loop=0,
)
print(f"\nSaved: {gif_path}  ({len(pil_frames)} frame(s))")